# Lab 8 — Actor–Critic for Heterogeneous Agents: Aiyagari and Krusell–Smith

**ECON 282E · Session 8 · November 12, 2026**

> Part B follows a method from research in progress by Feng, Han, Sargent and Zhu (8.B1). The code here is
> written for this course, at a small scale, on Session 6's economy; it carries none of their numbers.

Block A of Session 8 solved one household with an actor and a critic (8.A3). Block B asks the same two networks
to solve an *economy*, where the household's problem depends on what everyone else does. This lab runs both
of Block B's answers:

| Part | What you run | Deck |
|---|---|---|
| **A** | the answer key: Session 6's stationary equilibrium on the histogram grid, and a tilted start | 6.A2, 6.B2, 8.B1 |
| **B** | the Aiyagari household by actor–critic, with the whole histogram $\Gamma$ as part of its state | 8.A3, 8.B1 |
| **C** | Krusell–Smith by DeepHAM: a panel of households read through a generalised moment | 6.B1, 8.B2 |

**How the code is organised.** Part 0 writes five files to `./l08_code/` and imports them: three of Session 6's
solvers and the two Session 8 modules. Read them there, or in the cells; every important line carries a comment.
The later cells call their functions one at a time, in the order of the decks.

**Gates and the ledger.** Each part ends with a **gate**, a bound fixed before the run; a failed gate stops the
notebook. `LEDGER` holds the numbers of the reference run (`s08_lab_numbers.json`), so you can set yours beside them.

**Timing.** About five minutes on a laptop CPU from a clean kernel: Part B trains a short configuration (`LAB`,
about a minute and a half) and Part C trains DeepHAM's (`LIVE`, a few minutes). Set `USE_RECORDED = True` to
load the reference networks from `l08_assets/` instead; the cells then reproduce the ledger's numbers exactly.

In [ ]:
import os, sys, time, json
os.environ.setdefault("S08_THREADS", "4")
from pathlib import Path
os.makedirs("l08_code", exist_ok=True)
sys.path.insert(0, os.path.abspath("l08_code"))

import numpy as np
import matplotlib.pyplot as plt
import torch
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

USE_RECORDED = False            # True: load the reference networks instead of training
ASSETS = Path("l08_assets") if Path("l08_assets").exists() else Path("labs") / "l08_assets"

GATE = {
 "A_K_star_rel_max": 1e-08,
 "B2_autograd_vs_fd_max": 1e-07,
 "B5_K_gap_max": 0.01,
 "B6_path_ratio_max": 0.5,
 "C_K_rel_gap_max": 0.08,
 "C_euler_mean_max": 0.01,
 "C_abs_cev_max": 0.005,
 "R_recorded_rel_max": 1e-06
}

def gate(name, ok, detail=""):
    print(f"GATE {name}: {'pass' if ok else 'FAIL'}   {detail}")
    assert ok, f"gate {name} failed: {detail}"

def rel(a, b):
    return abs(a - b) / max(abs(b), 1e-300)

LEDGER = {'K_star': 5.67856769939,
 'r_star_pct': 2.3079906,
 'top5_mass': 0.000325,
 's06_K_200': 5.63936,
 's06_r_pct_200': 2.35592,
 'egm_euler_mean': 0.002569,
 'tilt_K_over_Kstar': {'-0.2': 0.8355, '-0.1': 0.9108, '+0.1': 1.107, '+0.2': 1.236},
 'live_seed0': {'euler_mean': 0.0016522952,
                'ap_gap_mean': 0.0029602721,
                'K_gap': 0.003633268,
                'path_worst_actor': 0.025692378,
                'path_worst_fixed': 0.09562958},
 'live_median': {'euler_mean': 0.001618,
                 'ap_gap_mean': 0.002994,
                 'K_gap': 0.003099,
                 'path_worst_actor': 0.02305,
                 'path_worst_fixed': 0.09563,
                 'seconds': 863.0},
 'live_seeds': 4,
 'noroll_K_gap': 0.08298,
 'noroll_euler': 0.00193,
 'lab_worst': {'euler_mean': 0.001683, 'K_gap': 0.002921, 'path_worst_actor': 0.02918, 'seconds': 63.4},
 'dh_seed0': {'euler_mean_abs': 0.0031049349, 'K_mean': 38.534895, 'cev_vs_classical': -0.00013654823},
 'dh_median': {'euler_mean_abs': 0.003239,
               'euler_p90_abs': 0.005114,
               'K_mean': 38.27,
               'cev_vs_classical': -0.0001234,
               'critic_r2_last': 0.7117,
               'seconds': 292.0},
 'dh_seeds': 4,
 'dh_classical_euler': 0.001111,
 'dh_classical_K_panels': 38.91,
 'start_path_worst': 0.05838,
 'ks_K_mean': 39.69348,
 'ks_kappa_bad': [0.123950378438, 0.965526647397],
 'ks_kappa_good': [0.137136914846, 0.963400896681]}
print("torch", torch.__version__, "| numpy", np.__version__, "| recorded networks:", USE_RECORDED)

---
## Part 0 — the code

Run these five cells once; each writes one file. They are long, and you do not need to read them before you start:
the parts below point at the function each step uses.

* `s06_ha.py` — Session 6's Aiyagari economy: EGM, Young's lottery, the stationary distribution, the equilibrium (6.A1–6.A2).
* `s06_paths.py` — Session 6's transition: households backward, the distribution forward (6.B2). Here it gives the
  **perfect-foresight path** back to the steady state, the answer key for Part B's paths.
* `s06_ks.py` — Session 6's Krusell–Smith household (6.B1). Part C rebuilds 6.B1's policy from it.
* `s08_ha_ac.py` — **Part B**: the economy's tensors, the two networks, the lottery in PyTorch, the critic and actor
  steps, the loop and the grades.
* `s08_deepham.py` — **Part C**: DeepHAM's three networks, the panel, the pool, the critic's rows, the three clocks.

In [ ]:
%%writefile l08_code/s06_ha.py
"""Session 6 -- the Aiyagari economy in plain numpy.

  6.A1  the household at fixed prices by the endogenous grid method (egm_step,
        solve_household), graded by the Euler residual between the nodes
        (euler_residuals); value function iteration and time iteration are in s06_vfi.py
  6.A2  from a policy to a population: Young's (2010) lottery, the transition
        matrix, the invariant distribution by iteration or by a linear solve; the
        capital market; the stationary equilibrium by bisection on r, which refuses
        to run without a sign change; five checks

Copied on 2026-10-22 from the AI-ECON Session 4 lab (session04_ha.py).  Changes: a
guard for a one-state chain in rouwenhorst, and lines re-wrapped so that the blocks
between '# <<name>>' markers fit on a slide.  Imports: numpy only.
"""
from __future__ import annotations

import time
from dataclasses import dataclass, replace
from math import comb

import numpy as np


# -------------------------------------------------------------------------------------
# Calibration
# -------------------------------------------------------------------------------------
@dataclass(frozen=True)
class Params:
    """Annual calibration used by every Aiyagari, transition and life-cycle block."""
    alpha: float = 0.33       # capital share
    beta: float = 0.96        # discount factor
    delta: float = 0.08       # depreciation
    sigma: float = 2.0        # CRRA
    rho_e: float = 0.90       # persistence of log labour productivity
    sd_innov: float = 0.20    # s.d. of its innovation
    n_e: int = 7              # Rouwenhorst states
    a_min: float = 0.0        # borrowing limit a' >= a_min
    a_max: float = 150.0
    n_a: int = 200
    curv: float = 2.5         # grid curvature: dense near the constraint
    tau_k: float = 0.0        # tax on capital income, rebated lump-sum

    @property
    def sd_uncond(self) -> float:
        return self.sd_innov / np.sqrt(1.0 - self.rho_e ** 2)


def u(c, sigma):
    return np.log(c) if sigma == 1.0 else c ** (1.0 - sigma) / (1.0 - sigma)


def rouwenhorst(n: int, rho: float, sd_uncond: float):
    """Rouwenhorst chain for log e' = rho log e + innovation.

    Returns (levels with mean one, Pi, pi).  n = 1 is a household without income
    risk: one level, Pi = [[1]] (used by the representative-agent check of 6.B2).
    """
    if n == 1:                                       # Session 6: the one-state chain
        return np.ones(1), np.ones((1, 1)), np.ones(1)
    p = (1.0 + rho) / 2.0
    Pi = np.array([[p, 1 - p], [1 - p, p]])
    for k in range(3, n + 1):
        new = np.zeros((k, k))
        new[:-1, :-1] += p * Pi
        new[:-1, 1:] += (1 - p) * Pi
        new[1:, :-1] += (1 - p) * Pi
        new[1:, 1:] += p * Pi
        new[1:-1, :] /= 2.0
        Pi = new
    psi = sd_uncond * np.sqrt(n - 1)
    pi = np.array([comb(n - 1, i) for i in range(n)], dtype=float) / 2.0 ** (n - 1)
    e = np.exp(np.linspace(-psi, psi, n))
    return e / (pi @ e), Pi, pi


def make_grid(p: Params) -> np.ndarray:
    return p.a_min + (p.a_max - p.a_min) * np.linspace(0.0, 1.0, p.n_a) ** p.curv


def firm(r: float, p: Params):
    """Capital per unit of labour and the wage at a pre-tax net return r (L = 1)."""
    kl = (p.alpha / (r + p.delta)) ** (1.0 / (1.0 - p.alpha))
    return kl, (1.0 - p.alpha) * kl ** p.alpha


def r_upper(p: Params) -> float:
    """The household's patience bounds the AFTER-tax return: (1 - tau_k) r < 1/beta - 1."""
    return (1.0 / p.beta - 1.0) / (1.0 - p.tau_k)


# -------------------------------------------------------------------------------------
# 6.A1 -- the household, by the endogenous grid method
# -------------------------------------------------------------------------------------
def egm_step(c_next, r_net_next, r_net, w, transfer, grid, e, Pi, p: Params):
    """One backward step: tomorrow's consumption policy -> today's (c, a')."""
    R_next, R = 1.0 + r_net_next, 1.0 + r_net                # gross returns tomorrow, today
    Euc = (c_next ** (-p.sigma)) @ Pi.T                      # E[u'(c') | e] with a' = each node
    c_end = (p.beta * R_next * Euc) ** (-1.0 / p.sigma)       # Euler, inverted: today's c given a'
    a_end = (c_end + grid[:, None] - w * e[None, :] - transfer) / R   # the a that affords it
    coh = R * grid[:, None] + w * e[None, :] + transfer       # cash on hand at the FIXED nodes
    c = np.empty_like(c_next)
    for q in range(e.size):                                   # one productivity level at a time
        c[:, q] = np.interp(grid, a_end[:, q], c_end[:, q])   # read c back onto the fixed grid
        binds = grid < a_end[0, q]                            # poorer than a_1q: a' = a_min ...
        c[binds, q] = coh[binds, q] - p.a_min                 # ... so c = cash on hand - a_min
    ap = np.clip(coh - c, p.a_min, grid[-1])                  # a' from the budget, kept on the grid
    return coh - ap, ap


def solve_household(r_net, w, transfer, grid, e, Pi, p: Params, c_init=None,
                    tol=1e-9, max_iter=5000):
    """Stationary household problem at fixed prices. Returns (a', c, sweeps, change)."""
    coh = (1.0 + r_net) * grid[:, None] + w * e[None, :] + transfer   # cash on hand, (n_a, n_e)
    c = np.maximum(coh - p.a_min if c_init is None else c_init.copy(), 1e-10)  # guess: eat it all
    diff, it = np.inf, 0
    for it in range(1, max_iter + 1):                          # one sweep = one EGM step (K)
        c_new, ap = egm_step(c, r_net, r_net, w, transfer, grid, e, Pi, p)  # same prices tomorrow
        diff = float(np.abs(c_new - c).max())                  # how far the policy moved
        c = c_new
        if diff < tol:                                         # a fixed point of K: stop
            break
    return ap, c, it, diff


def euler_residuals(ap, c, r_net, w, transfer, grid, e, Pi, p: Params, n_off=3):
    """log10 |1 - c_Euler/c| at OFF-grid asset points where the constraint is slack.

    Returns (assets, residual matrix, slack mask). Points are placed strictly between grid nodes.
    """
    R = 1.0 + r_net
    fr = (np.arange(1, n_off + 1) / (n_off + 1.0))[None, :]
    a_off = (grid[:-1, None] + fr * np.diff(grid)[:, None]).ravel()
    res = np.full((a_off.size, e.size), np.nan)
    slack = np.zeros_like(res, dtype=bool)
    for q in range(e.size):
        c_here = np.interp(a_off, grid, c[:, q])
        ap_here = R * a_off + w * e[q] + transfer - c_here
        Euc = np.zeros_like(a_off)
        for qn in range(e.size):
            Euc += Pi[q, qn] * np.interp(ap_here, grid, c[:, qn]) ** (-p.sigma)
        c_euler = (p.beta * R * Euc) ** (-1.0 / p.sigma)
        slack[:, q] = (ap_here > p.a_min + 1e-8) & (ap_here < grid[-1])
        res[:, q] = np.log10(np.abs(1.0 - c_euler / c_here) + 1e-16)
    return a_off, res, slack


# -------------------------------------------------------------------------------------
# 6.A2 -- from a policy to a population
# -------------------------------------------------------------------------------------
def lottery(ap, grid):
    """Young (2010): split the mass at a' between the two neighbouring nodes."""
    a1 = np.clip(ap, grid[0], grid[-1])                        # a' inside [a_1, a_N]
    j = np.clip(np.searchsorted(grid, a1, side="right") - 1, 0, grid.size - 2)  # a_j <= a' < a_j+1
    return j, (a1 - grid[j]) / (grid[j + 1] - grid[j])         # j, and omega: the share sent up


def push_forward(lam, j, w_hi, Pi):
    """One application of the operator on measures: lambda_t -> lambda_{t+1}."""
    na, ne = lam.shape
    out = np.zeros_like(lam)
    for q in range(ne):                                        # today's productivity level
        m = lam[:, q]                                          # mass at each node, given e_q
        tmp = (np.bincount(j[:, q], weights=m * (1.0 - w_hi[:, q]), minlength=na)   # 1-omega to a_j
               + np.bincount(j[:, q] + 1, weights=m * w_hi[:, q], minlength=na))    # omega to a_j+1
        out += np.outer(tmp, Pi[q, :])                         # then e_q -> e_q' with Pi[q, q']
    return out


def transition_matrix(j, w_hi, Pi):
    """The same operator as a dense matrix over (a, e) pairs, row-stochastic."""
    na, ne = j.shape
    P = np.zeros((na * ne, na * ne))                           # one row, one column per (a_i, e_q)
    rows = np.arange(na)
    for q in range(ne):                                        # today's e_q ...
        for qn in range(ne):                                   # ... and tomorrow's e_q'
            lo, hi = j[:, q] * ne + qn, (j[:, q] + 1) * ne + qn    # the two columns k'
            np.add.at(P, (rows * ne + q, lo), (1.0 - w_hi[:, q]) * Pi[q, qn])   # row k = ne i + q
            np.add.at(P, (rows * ne + q, hi), w_hi[:, q] * Pi[q, qn])          # omega Pi[q, q']
    return P


def stationary_distribution(ap, grid, Pi, pi, method="solve", lam_init=None,
                            tol=1e-11, max_iter=200000):
    """Invariant distribution over (a, e): by a linear solve, or by iteration."""
    na, ne = ap.shape
    j, w_hi = lottery(ap, grid)                                # where the policy sends each node
    if method == "solve":                                      # Way 2: (P' - I) lambda = 0
        A = transition_matrix(j, w_hi, Pi).T - np.eye(na * ne)   # n_a n_e equations, rank one less
        A[-1, :] = 1.0                                         # replace one equation by sum = 1
        b = np.zeros(na * ne); b[-1] = 1.0                     # right-hand side: 0, ..., 0, 1
        lam = np.maximum(np.linalg.solve(A, b), 0.0).reshape(na, ne)   # one dense solve
        return lam / lam.sum(), 1, 0.0
    lam = np.zeros((na, ne)); lam[0, :] = pi                   # everyone starts at the limit
    if lam_init is not None:                                   # ... or at a warm start
        lam = lam_init.copy()
    diff, it = np.inf, 0
    for it in range(1, max_iter + 1):                          # Way 1: lambda <- lambda P
        new = push_forward(lam, j, w_hi, Pi)
        diff = float(np.abs(new - lam).max())
        lam = new
        if diff < tol:                                         # it no longer moves: invariant
            break
    return lam / lam.sum(), it, diff


def simulate_panel(ap, grid, Pi, pi, n=10000, T=1000, burn=500, seed=0):
    """Monte Carlo: N households, each a draw of e' from Pi and a' = a'(a, e) read exactly."""
    rng = np.random.default_rng(seed)
    cum = np.cumsum(Pi, axis=1); cum[:, -1] = 1.0              # rows of Pi as cumulative sums
    a = np.zeros(n)                                            # everyone starts at the limit ...
    q = rng.choice(pi.size, size=n, p=pi)                      # ... with e drawn from pi
    K = np.empty(T - burn)
    for t in range(T):
        for s in range(pi.size):                               # by productivity group
            here = q == s
            a[here] = np.interp(a[here], grid, ap[:, s])       # a' between the nodes: no lottery
        q = (rng.random(n)[:, None] > cum[q]).sum(axis=1)      # one uniform draws e' from Pi[q, :]
        if t >= burn:
            K[t - burn] = a.mean()                             # K_t = mean assets of the sample
    return K, a, q


def gini(values, mass) -> float:
    m = np.asarray(mass, dtype=float).ravel(); x = np.asarray(values, dtype=float).ravel()
    order = np.argsort(x); x, m = x[order], m[order] / m.sum()
    S = np.cumsum(m * x)
    if S[-1] <= 0:
        return 0.0
    L = np.concatenate([[0.0], S / S[-1]]); F = np.concatenate([[0.0], np.cumsum(m)])
    return float(1.0 - np.sum((F[1:] - F[:-1]) * (L[1:] + L[:-1])))


def top_share(values, mass, top=0.10) -> float:
    m = np.asarray(mass, dtype=float).ravel(); x = np.asarray(values, dtype=float).ravel()
    order = np.argsort(x)[::-1]; x, m = x[order], m[order] / m.sum()
    cum = np.cumsum(m); k = np.searchsorted(cum, top)
    held = (m[:k] * x[:k]).sum() + (top - (cum[k - 1] if k else 0.0)) * x[k]
    return float(held / (m * x).sum())


# -------------------------------------------------------------------------------------
# 6.A2 -- the stationary equilibrium
# -------------------------------------------------------------------------------------
def economy(p: Params):
    e, Pi, pi = rouwenhorst(p.n_e, p.rho_e, p.sd_uncond)
    return make_grid(p), e, Pi, pi


def capital_supply(r, p: Params, grid, e, Pi, pi, c_init=None, dist="iterate",
                   lam_init=None):
    """Households' capital at a pre-tax return r; the tax is rebated lump-sum."""
    kl, w = firm(r, p)                                         # K/L the firm wants at r, and w
    r_net, transfer = (1.0 - p.tau_k) * r, p.tau_k * r * kl   # after-tax return; the rebate
    ap, c, it, diff = solve_household(r_net, w, transfer, grid, e, Pi, p,
                                      c_init=c_init)          # 6.A1: the policy at these prices
    lam, it_d, _ = stationary_distribution(ap, grid, Pi, pi, method=dist,
                                           lam_init=lam_init)  # Part 3: the population
    Ks = float((lam.sum(axis=1) * grid).sum())                 # K supplied = mean assets
    return dict(r=r, r_net=r_net, w=w, transfer=transfer, K_demand=kl, K_supply=Ks,
                ap=ap, c=c, lam=lam, egm_sweeps=it, dist_sweeps=it_d)


def solve_equilibrium(p: Params, r_lo=None, r_hi=None, tol=1e-7, dist="iterate",
                      verbose=False):
    """Bisection on the pre-tax return. Refuses to run without a sign change at the ends.

    The policy and the distribution are warm-started from the previous price. dist="solve"
    uses the dense linear solve instead of iteration: the same fixed point, slower here.
    """
    t0 = time.time()
    grid, e, Pi, pi = economy(p)
    r_lo = -p.delta + 0.02 if r_lo is None else r_lo
    r_hi = r_upper(p) - 1e-5 if r_hi is None else r_hi
    lo = capital_supply(r_lo, p, grid, e, Pi, pi, dist=dist)   # both ends of the bracket
    hi = capital_supply(r_hi, p, grid, e, Pi, pi, dist=dist)
    d_lo, d_hi = lo["K_supply"] - lo["K_demand"], hi["K_supply"] - hi["K_demand"]  # excess supply
    if not (d_lo < 0.0 < d_hi):                                # no sign change: refuse to run
        raise RuntimeError(f"no sign change on [{r_lo:.5f}, {r_hi:.5f}]: "
                           f"excess supply {d_lo:.4f}, {d_hi:.4f}")
    sol, n = hi, 0
    while r_hi - r_lo > tol:                                   # halve until the bracket is tiny
        n += 1
        sol = capital_supply(0.5 * (r_lo + r_hi), p, grid, e, Pi, pi,   # the midpoint ...
                             c_init=sol["c"], dist=dist, lam_init=sol["lam"])  # ... warm-started
        d = sol["K_supply"] - sol["K_demand"]                  # excess supply there
        if verbose:
            print(f"   {n:2d}  r = {sol['r']:.6f}   excess supply = {d:+.6f}")
        if d > 0:                                              # supply > demand: r is too high
            r_hi = sol["r"]
        else:                                                  # demand > supply: r is too low
            r_lo = sol["r"]
    sol.update(grid=grid, e=e, Pi=Pi, pi=pi, bisections=n, seconds=time.time() - t0,
               params=p, clearing_residual=abs(sol["K_supply"] - sol["K_demand"])
               / sol["K_demand"], bracket=(r_lo, r_hi), sign_change=(d_lo, d_hi))
    return sol


def excess_demand_update(p: Params, r0=0.02, eta=0.02, tol=1e-5, max_iter=100,
                         dist="iterate"):
    """Move r by the excess demand for capital over K_d + K_s: a step is at most eta."""
    grid, e, Pi, pi = economy(p)
    r_lo, r_hi = -p.delta + 0.01, r_upper(p) - 1e-5            # outside: no bounded solution
    r, sol, path = r0, None, []
    for n in range(1, max_iter + 1):
        sol = capital_supply(r, p, grid, e, Pi, pi, dist=dist,  # households at r ...
                             c_init=None if sol is None else sol["c"],
                             lam_init=None if sol is None else sol["lam"])   # ... warm-started
        Kd, Ks = sol["K_demand"], sol["K_supply"]
        gap = (Kd - Ks) / (Kd + Ks)                            # excess demand, in (-1, 1)
        path.append((r, gap))
        if abs(Kd - Ks) / Kd < tol:                            # the market clears: stop
            break
        r = float(np.clip(r + eta * gap, r_lo, r_hi))          # demand > supply: r rises
    return dict(r=r, iterations=n, path=path, K_demand=Kd, K_supply=Ks, sol=sol)


def five_checks(sol) -> dict:
    """The five things to check before believing a stationary equilibrium."""
    p, grid, lam = sol["params"], sol["grid"], sol["lam"]
    a_off, res, slack = euler_residuals(sol["ap"], sol["c"], sol["r_net"], sol["w"],
                                        sol["transfer"], grid, sol["e"], sol["Pi"], p)
    wts = np.repeat(lam[:-1, :] / 3.0, 3, axis=0)            # mass near each off-grid point
    wealth = np.repeat(grid[:, None], lam.shape[1], 1)
    return dict(
        mass_error=float(abs(lam.sum() - 1.0)),
        mass_at_top=float(lam[-1, :].sum()),
        euler_log10_max=float(np.nanmax(np.where(slack, res, np.nan))),
        euler_log10_mean_weighted=float(np.nansum(np.where(slack, res, 0.0) * wts)
                                        / np.sum(wts * slack)),
        clearing_residual=float(sol["clearing_residual"]),
        share_constrained=float(lam[0, :].sum()),
        gini_wealth=gini(wealth, lam),
        gini_earnings=gini(sol["e"], sol["pi"]),
        top10_share=top_share(wealth, lam, 0.10),
        top1_share=top_share(wealth, lam, 0.01),
    )

In [ ]:
%%writefile l08_code/s06_paths.py
"""Session 6 -- paths and cohorts, in plain numpy.

  6.B2  the transition path of the infinitely-lived economy after an unanticipated,
        permanent tax on capital income (an "MIT shock"), with the welfare arithmetic
        that a comparison of two steady states gets wrong; and the stationary life-cycle
        (overlapping-generations) economy with a pay-as-you-go pension whose budget is
        balanced every period by a payroll tax.  In finite lives 1/beta - 1 is NOT an
        upper bound on the interest rate, so the life-cycle bracket is wide.

Copied on 2026-10-22 from the AI-ECON Session 4 lab (session04_paths.py), without its
recording code; lines between '# <<name>>' markers re-wrapped to fit a slide.
Imports: numpy and s06_ha.
"""
from __future__ import annotations

import time
from dataclasses import replace

import numpy as np

from s06_ha import (Params, economy, egm_step, firm, gini, lottery, push_forward,
                    solve_equilibrium, u)


# -------------------------------------------------------------------------------------
# Welfare arithmetic
# -------------------------------------------------------------------------------------
def steady_state_value(c, ap, grid, Pi, p: Params, tol=1e-12, max_iter=50000):
    """Value of following a stationary policy for ever: iterate V = u(c) + beta E[V(a', e')]."""
    flow = u(c, p.sigma)
    V = flow / (1.0 - p.beta)
    for _ in range(max_iter):
        EV = V @ Pi.T
        cont = np.empty_like(V)
        for q in range(Pi.shape[0]):
            cont[:, q] = np.interp(ap[:, q], grid, EV[:, q])
        V_new = flow + p.beta * cont
        done = float(np.abs(V_new - V).max()) < tol
        V = V_new
        if done:
            break
    return V


def values_along(c_path, ap_path, grid, Pi, p: Params, V_final):
    """Values along a path, backward from V_T = V_final: V_t = u(c_t) + beta E[V_{t+1}(a'_t, e')]."""
    T = len(c_path) - 1
    V = [None] * (T + 1)
    V[T] = V_final
    for t in range(T - 1, -1, -1):
        EV = V[t + 1] @ Pi.T
        cont = np.empty_like(EV)
        for q in range(Pi.shape[0]):
            cont[:, q] = np.interp(ap_path[t][:, q], grid, EV[:, q])
        V[t] = u(c_path[t], p.sigma) + p.beta * cont
    return V


def cev(V_new, V_old, p: Params):
    """Consumption-equivalent variation g: scale the OLD consumption plan by (1 + g) to match V_new.

    CRRA: V(c (1+g)) = (1+g)^(1-sigma) V(c), so g = (V_new / V_old)^(1/(1-sigma)) - 1. With sigma > 1
    both values are negative, the ratio is positive, and a less negative V_new gives g > 0.
    Log utility: V(c (1+g)) = V(c) + log(1+g) / (1-beta).
    """
    V_new, V_old = np.asarray(V_new, dtype=float), np.asarray(V_old, dtype=float)
    if p.sigma == 1.0:
        return np.exp((1.0 - p.beta) * (V_new - V_old)) - 1.0
    ratio = V_new / V_old
    if np.any(ratio <= 0.0):
        raise ValueError("values of opposite sign: the CRRA formula does not apply")
    return ratio ** (1.0 / (1.0 - p.sigma)) - 1.0


def half_life(path, x0, x1) -> float:
    """Periods until the path has covered half of the distance from x0 to x1 (linear in between)."""
    if abs(x1 - x0) < 1e-14:
        return float("nan")
    frac = (np.asarray(path) - x0) / (x1 - x0)
    idx = np.where(frac >= 0.5)[0]
    if idx.size == 0:
        return float("nan")
    k = int(idx[0])
    if k == 0:
        return 0.0
    return float(k - 1 + (0.5 - frac[k - 1]) / (frac[k] - frac[k - 1]))


def _decile_means(g, lam, n=10):
    """Mean of g within each wealth decile of the distribution lam (mass split exactly at the edges)."""
    mass_a = lam.sum(axis=1)
    g_a = (g * lam).sum(axis=1) / np.maximum(mass_a, 1e-300)
    cum = np.concatenate([[0.0], np.cumsum(mass_a)])
    out = []
    for d in range(n):
        lo, hi = d / n, (d + 1) / n
        share = np.clip(np.minimum(cum[1:], hi) - np.maximum(cum[:-1], lo), 0.0, None)
        out.append(float((share * g_a).sum() / share.sum()))
    return out


# -------------------------------------------------------------------------------------
# 6.B2 -- the transition path
# -------------------------------------------------------------------------------------
def path_prices(K, p: Params, tau_k: float):
    """Prices and the rebate along a path of capital (L = 1): r, (1 - tau) r, w, tau r K."""
    r = p.alpha * K ** (p.alpha - 1.0) - p.delta
    return r, (1.0 - tau_k) * r, (1.0 - p.alpha) * K ** p.alpha, tau_k * r * K


def backward_policies(r_net, w, transfer, c_final, grid, e, Pi, p: Params):
    """Households backward from the terminal policy: one EGM step per period."""
    T = len(r_net) - 1
    c_path, ap_path = [None] * (T + 1), [None] * (T + 1)
    c_path[T] = c_final
    for t in range(T - 1, -1, -1):
        c_path[t], ap_path[t] = egm_step(c_path[t + 1], r_net[t + 1], r_net[t],
                                         w[t], transfer[t], grid, e, Pi, p)
    return c_path, ap_path


def forward_capital(lam0, ap_path, grid, Pi, K0, c_path=None):
    """Distribution forward from lam0: capital supplied, lambda_T, and (given the
    consumption policies) aggregate consumption in periods 0..T-1."""
    T = len(ap_path) - 1
    Ks, C = np.empty(T + 1), np.full(T + 1, np.nan)
    Ks[0] = K0
    lam = lam0
    for t in range(T):
        if c_path is not None:
            C[t] = float((lam * c_path[t]).sum())
        j, w_hi = lottery(ap_path[t], grid)
        lam = push_forward(lam, j, w_hi, Pi)
        Ks[t + 1] = float((lam.sum(axis=1) * grid).sum())
    return Ks, lam, C


def steady_state_after_tax(p_new: Params, ss_old: dict, dist: str = "iterate") -> dict:
    """The stationary equilibrium under the tax, from `solve_equilibrium` (sign change
    asserted there).

    A tax on capital income lowers capital and raises the pre-tax return, but not beyond
    the rate that restores the old AFTER-tax return: there households would hold about as
    much as before while firms want less. So [r_old, r_old / (1 - tau_k)] is tried first,
    and the wide after-tax bracket is the fallback if the sign test refuses.
    """
    if p_new.tau_k > 0.0:
        try:
            return solve_equilibrium(p_new, r_lo=ss_old["r"],
                                     r_hi=ss_old["r"] / (1.0 - p_new.tau_k), dist=dist)
        except RuntimeError:
            pass
    return solve_equilibrium(p_new, dist=dist)


def transition_path(p: Params = Params(), tau_k: float = 0.20, T: int = 150,
                    damping: float = 0.5, tol: float = 1e-5, max_outer: int = 300,
                    ss_old=None, ss_new=None, K_init=None, dist: str = "iterate",
                    verbose: bool = False) -> dict:
    """Perfect-foresight path after an unanticipated, permanent tax on capital income.

    Timing. Up to t = -1 the economy rests in the stationary equilibrium with no tax. At the
    start of t = 0 the tax tau_k is announced, applies at once and stays for ever. K_0 and
    lambda_0 are predetermined: they are the old steady state's. In EVERY period t >= 0
    households earn (1 - tau_k) r_t on their assets and receive the rebate tau_k r_t K_t;
    firms pay r_t and w_t at K_t. Households know the whole path from t = 0 on. Period T
    behaves as the new steady state.

    Algorithm. Guess {K_t}; prices from K_t; households backward from the new steady
    state's policy; the distribution forward from lambda_0; compare the capital households
    supply with the guess; damped update; stop when max_t |K^s_t - K_t| / K_t < tol.
    """
    t0 = time.time()
    p_old, p_new = replace(p, tau_k=0.0), replace(p, tau_k=tau_k)
    if ss_old is None:
        ss_old = solve_equilibrium(p_old, dist=dist)
    if ss_new is None:
        ss_new = steady_state_after_tax(p_new, ss_old, dist=dist)
    t_ss = time.time() - t0
    grid, e, Pi = ss_old["grid"], ss_old["e"], ss_old["Pi"]
    K_old, K_new = ss_old["K_supply"], ss_new["K_supply"]
    lam0 = ss_old["lam"]

    tt = np.arange(T + 1)
    if K_init is None:
        K = K_new + (K_old - K_new) * np.exp(-tt / 12.0)
    else:                                                  # warm start from a shorter path
        K = np.concatenate([K_init, np.full(max(0, T + 1 - len(K_init)), K_init[-1])])[:T + 1].copy()
    K[0] = K_old

    gap, outer, history = np.inf, 0, []
    t1 = time.time()
    for outer in range(1, max_outer + 1):
        r, r_net, w, transfer = path_prices(K, p_new, tau_k)       # prices from the guess
        c_path, ap_path = backward_policies(r_net, w, transfer, ss_new["c"],
                                            grid, e, Pi, p_new)     # households backward
        Ks, lam_T, C = forward_capital(lam0, ap_path, grid, Pi, K_old,
                                       c_path)                      # distribution forward
        resid = np.abs(Ks - K) / K
        gap = float(resid.max())
        history.append(gap)
        if verbose:
            print(f"   outer {outer:3d}   max |K^s - K| / K = {gap:.3e}")
        if gap < tol:
            break
        K = (1.0 - damping) * K + damping * Ks                     # damped update
        K[0] = K_old                                               # K_0 is given
    converged = gap < tol
    t_path = time.time() - t1

    r, r_net, w, transfer = path_prices(K, p_new, tau_k)
    Y = K ** p_new.alpha
    # goods add up: C_t + K_{t+1} = (1 - delta) K_t + Y_t, an identity only if budgets,
    # rebates and the lottery are all consistent (and no mass is clipped at the top)
    resources = np.abs(C[:-1] + Ks[1:] - (1.0 - p_new.delta) * K[:-1] - Y[:-1]) / Y[:-1]
    out = dict(params=p_new, tau_k=tau_k, T=T, damping=damping, tol=tol, outer_iterations=outer,
               converged=bool(converged), gap_history=history,
               K=K, K_supply=Ks, r=r, r_net=r_net, w=w, transfer=transfer,
               residual=resid, max_residual=gap, C=C, Y=Y,
               resource_residual=resources, max_resource_residual=float(resources.max()),
               lam_T=lam_T, terminal_L1=float(np.abs(lam_T - ss_new["lam"]).sum()),
               terminal_K_gap=float(abs(K[-1] - K_new) / K_new),
               K_old=K_old, K_new=K_new, K_change_pct=100.0 * (K_new / K_old - 1.0),
               r_old=ss_old["r"], r_new=ss_new["r"], r_net_old=ss_old["r_net"],
               r_net_new=ss_new["r_net"], w_old=ss_old["w"], w_new=ss_new["w"],
               half_life_K=half_life(K, K_old, K_new),
               ss_old=ss_old, ss_new=ss_new, c_path=c_path, ap_path=ap_path,
               seconds_steady_states=t_ss, seconds_path=t_path, seconds=time.time() - t0)
    return out


def transition_welfare(tr: dict) -> dict:
    """Who gains. CEV at t = 0 for every (a, e) along the path, against staying in the old
    steady state -- and the number a comparison of the two steady states would have reported."""
    p, ss0, ss1 = tr["params"], tr["ss_old"], tr["ss_new"]
    grid, Pi = ss0["grid"], ss0["Pi"]
    lam0, lam1 = ss0["lam"], ss1["lam"]
    V_old = steady_state_value(ss0["c"], ss0["ap"], grid, Pi, p)
    V_new = steady_state_value(ss1["c"], ss1["ap"], grid, Pi, p)
    V_path = values_along(tr["c_path"], tr["ap_path"], grid, Pi, p, V_new)

    g = cev(V_path[0], V_old, p)                        # state by state, the path included
    g_same_state = cev(V_new, V_old, p)                 # same (a, e), the path ignored
    W_old, W_new = float((V_old * lam0).sum()), float((V_new * lam1).sum())
    W_path = float((V_path[0] * lam0).sum())
    wealth0 = np.repeat(grid[:, None], lam0.shape[1], 1)
    wealth1 = np.repeat(grid[:, None], lam1.shape[1], 1)
    return dict(
        cev_state=g,
        cev_mean_pct=100.0 * float((g * lam0).sum()),
        cev_utilitarian_pct=100.0 * float(cev(W_path, W_old, p)),
        share_worse_off_pct=100.0 * float(lam0[g < 0.0].sum()),
        cev_steady_states_own_distribution_pct=100.0 * float(cev(W_new, W_old, p)),
        cev_steady_states_same_state_mean_pct=100.0 * float((g_same_state * lam0).sum()),
        cev_by_wealth_decile_pct=[100.0 * x for x in _decile_means(g, lam0)],
        cev_by_productivity_pct=[100.0 * float((g[:, q] * lam0[:, q]).sum() / lam0[:, q].sum())
                                 for q in range(lam0.shape[1])],
        cev_min_pct=100.0 * float(g[lam0 > 1e-12].min()),
        cev_max_pct=100.0 * float(g[lam0 > 1e-12].max()),
        gini_old=gini(wealth0, lam0), gini_new=gini(wealth1, lam1),
        V_old=V_old, V_new=V_new, V_path0=V_path[0])


def cev_sanity(ss, p: Params, T: int = 25) -> dict:
    """Two checks of the welfare arithmetic. (1) No change, no gain: along a path that IS the
    steady state the CEV is zero. (2) Scale the old plan by (1 + g0): the formula returns g0."""
    grid, Pi = ss["grid"], ss["Pi"]
    V = steady_state_value(ss["c"], ss["ap"], grid, Pi, p)
    V_path = values_along([ss["c"]] * (T + 1), [ss["ap"]] * (T + 1), grid, Pi, p, V)
    no_change = float(np.abs(cev(V_path[0], V, p)).max())
    g0 = 0.0123
    scaled = ((1.0 + g0) ** (1.0 - p.sigma) * V if p.sigma != 1.0
              else V + np.log(1.0 + g0) / (1.0 - p.beta))
    recovered = float(np.abs(cev(scaled, V, p) - g0).max())
    assert no_change < 1e-8, f"CEV of no change is {no_change:.2e}, not zero"
    assert recovered < 1e-12, f"CEV formula misses a known scaling by {recovered:.2e}"
    return dict(no_change_max_abs_cev=no_change, known_scaling_error=recovered)


# -------------------------------------------------------------------------------------
# The stationary life-cycle economy with a pay-as-you-go pension financed by a payroll tax
# -------------------------------------------------------------------------------------
def olg_setup(p: Params = Params(), n_cohorts: int = 60, n_work: int = 45, a_max: float = 80.0,
              n_a: int = 250):
    """Sixty equal cohorts (ages 21-80), forty-five working years, a hump-shaped efficiency profile."""
    p_olg = replace(p, tau_k=0.0, a_min=0.0, a_max=a_max, n_a=n_a)
    grid, e, Pi, pi = economy(p_olg)
    i = np.arange(n_cohorts, dtype=float)
    eff = np.exp(0.09 * i - 0.0013 * i ** 2)
    eff[n_work:] = 0.0
    eff[:n_work] /= eff[:n_work].mean()                   # mean one over the working years
    L = float(eff.sum() / n_cohorts)                      # effective labour per person (E[e] = 1)
    return dict(p=p_olg, grid=grid, e=e, Pi=Pi, pi=pi, eff=eff, L=L, I=n_cohorts, I_R=n_work,
                ages=21 + np.arange(n_cohorts))


def payroll_tax(replacement: float, I: int, I_R: int) -> float:
    """Pension b = replacement x the average NET wage income of workers, paid from a payroll
    tax tau_p.  Budget: tau_p w (I_R / I) = b (I - I_R) / I with b = replacement (1 - tau_p) w."""
    return replacement * (I - I_R) / (I_R + replacement * (I - I_R))


def olg_at_r(r: float, replacement: float, s: dict) -> dict:
    """Everything at a candidate interest rate: households backward over ages, cohorts
    forward, aggregates."""
    p, grid, e, Pi, pi, eff, I, I_R = s["p"], s["grid"], s["e"], s["Pi"], s["pi"], s["eff"], s["I"], s["I_R"]
    kl, w = firm(r, p)
    tau_p = payroll_tax(replacement, I, I_R)
    b = replacement * (1.0 - tau_p) * w                   # average gross wage income of workers is w
    na, ne = grid.size, e.size
    c_age, ap_age = [None] * I, [None] * I
    c_last = np.maximum((1.0 + r) * grid + b, 1e-10)     # the last year: eat everything
    c_age[I - 1] = np.repeat(c_last[:, None], ne, axis=1)
    ap_age[I - 1] = np.zeros((na, ne))
    for i in range(I - 2, -1, -1):                       # one pass over ages, no fixed point
        wage_i, pension_i = ((1.0 - tau_p) * w * eff[i], 0.0) if i < I_R else (0.0, b)
        c_age[i], ap_age[i] = egm_step(c_age[i + 1], r, r, wage_i, pension_i,
                                       grid, e, Pi, p)
        c_age[i] = np.maximum(c_age[i], 1e-10)          # no pension, no assets: eats nothing
    lam = np.zeros((I, na, ne))
    lam[0, 0, :] = pi / I                                # newborns: no assets, e from pi
    for i in range(I - 1):                               # the cohorts ARE the distribution
        j, w_hi = lottery(ap_age[i], grid)
        lam[i + 1] = push_forward(lam[i], j, w_hi, Pi)
    K_supply = float((lam.sum(axis=2) * grid[None, :]).sum())
    K_demand = kl * s["L"]
    gross_wages = float(sum((lam[i].sum(axis=0) * e).sum() * w * eff[i] for i in range(I_R)))
    pensions = float(sum(lam[i].sum() * b for i in range(I_R, I)))
    return dict(r=r, w=w, kl=kl, tau_p=tau_p, pension=b, replacement=replacement, K_supply=K_supply,
                K_demand=K_demand, c_age=c_age, ap_age=ap_age, lam=lam,
                budget_residual=abs(tau_p * gross_wages - pensions),
                bequests=float((lam[I - 1] * ap_age[I - 1]).sum()))


def solve_olg(p: Params = Params(), replacement: float = 0.40, r_lo: float = None, r_hi: float = 0.30,
              tol: float = 1e-9, setup: dict = None, verbose: bool = False) -> dict:
    """Bisection on r over a WIDE bracket: in finite lives r may exceed 1/beta - 1.
    Refuses to run without a sign change of excess capital supply at the two ends."""
    t0 = time.time()
    s = olg_setup(p) if setup is None else setup
    p_olg = s["p"]
    r_lo = -p_olg.delta + 0.01 if r_lo is None else r_lo
    lo, hi = olg_at_r(r_lo, replacement, s), olg_at_r(r_hi, replacement, s)
    d_lo, d_hi = lo["K_supply"] - lo["K_demand"], hi["K_supply"] - hi["K_demand"]
    if not (d_lo < 0.0 < d_hi):
        raise RuntimeError(f"no sign change on [{r_lo:.4f}, {r_hi:.4f}]: excess supply {d_lo:.4f}, {d_hi:.4f}")
    a, b_, n = r_lo, r_hi, 0
    sol = hi
    while b_ - a > tol:
        n += 1
        sol = olg_at_r(0.5 * (a + b_), replacement, s)
        d = sol["K_supply"] - sol["K_demand"]
        if verbose:
            print(f"   {n:2d}  r = {sol['r']:.7f}   excess supply = {d:+.7f}")
        if d > 0:
            b_ = sol["r"]
        else:
            a = sol["r"]
    lam, grid, e, I = sol["lam"], s["grid"], s["e"], s["I"]
    mass_age = lam.sum(axis=(1, 2))
    assets = (lam.sum(axis=2) * grid[None, :]).sum(axis=1) / mass_age
    cons = np.array([(lam[i] * sol["c_age"][i]).sum() for i in range(I)]) / mass_age
    gross_lab = np.array([(lam[i].sum(axis=0) * e).sum() * sol["w"] * s["eff"][i] for i in range(I)]) / mass_age
    net_income = (1.0 - sol["tau_p"]) * gross_lab + np.where(np.arange(I) >= s["I_R"], sol["pension"], 0.0)
    K, L = sol["K_supply"], s["L"]
    Y = K ** p_olg.alpha * L ** (1.0 - p_olg.alpha)
    wealth = np.repeat(np.repeat(grid[None, :, None], I, axis=0), e.size, axis=2)
    sol.update(setup=s, bisections=n, bracket=(r_lo, r_hi), sign_change=(d_lo, d_hi),
               clearing_residual=abs(sol["K_supply"] - sol["K_demand"]) / sol["K_demand"],
               mass_error=float(abs(lam.sum() - 1.0)), mass_at_top=float(lam[:, -1, :].sum()),
               K=K, L=L, Y=Y, K_over_Y=K / Y, C=float((cons * mass_age).sum()),
               resource_residual=float(abs(Y - p_olg.delta * K - (cons * mass_age).sum()) / Y),
               r_vs_patience=sol["r"] - (1.0 / p_olg.beta - 1.0),
               profile_age=s["ages"], profile_assets=assets, profile_consumption=cons,
               profile_gross_labour_income=gross_lab, profile_net_income=net_income,
               peak_asset_age=int(s["ages"][int(np.argmax(assets))]),
               gini_wealth=gini(wealth, lam), seconds=time.time() - t0)
    assert sol["budget_residual"] < 1e-10, f"pension budget does not balance: {sol['budget_residual']:.2e}"
    assert sol["bequests"] == 0.0, "assets are left at the last age"
    return sol


def olg_comparison(p: Params = Params(), rates=(0.40, 0.0), setup: dict = None) -> list:
    """The same economy under different replacement rates: how a pension moves K and r."""
    s = olg_setup(p) if setup is None else setup
    rows = []
    for rep in rates:
        o = solve_olg(p, replacement=rep, setup=s)
        rows.append(dict(replacement=rep, tau_p=o["tau_p"], r_pct=100.0 * o["r"], K=o["K"], Y=o["Y"],
                         K_over_Y=o["K_over_Y"], w=o["w"], pension=o["pension"],
                         peak_asset_age=o["peak_asset_age"], gini_wealth=o["gini_wealth"],
                         clearing_residual=o["clearing_residual"], solution=o))
    return rows


def olg_scan(replacement: float, setup: dict, rates=None) -> dict:
    """Capital supplied and demanded on a grid of interest rates -- the picture behind the wide bracket."""
    rates = np.linspace(0.0, 0.12, 13) if rates is None else np.asarray(rates)
    pts = [olg_at_r(float(r), replacement, setup) for r in rates]
    return dict(r=rates, K_supply=np.array([x["K_supply"] for x in pts]),
                K_demand=np.array([x["K_demand"] for x in pts]))

In [ ]:
%%writefile l08_code/s06_ks.py
"""Session 6 -- Krusell and Smith (1998) in plain numpy.

The economy is the one every comparison in the literature uses: Krusell and Smith's, in
the parameterisation of Den Haan, Judd and Juillard (JEDC 2010, "Computational suite of
models with heterogeneous agents"), where the unemployed also receive a benefit financed
by a tax on the employed.

  solve_ks       the household by the endogenous grid method on (a, employment, K, z)
                 under a log-linear perceived law of motion; non-stochastic simulation
                 of the histogram with Young's (2010) lottery; regression by aggregate
                 state; damped update of the coefficients
  diagnose_ks    R^2, and Den Haan's (2010) dynamic forecast error on a FRESH shock
                 path: the rule is iterated forward from one initial K and never
                 re-synchronised with the simulation
  The value-function route for the same household is s06_vfi.vfi_household_ks.

The two restrictions on job-finding when the aggregate state switches (1.25 and 0.75)
are Krusell and Smith's; the matrix built from them reproduces the published table of
Den Haan, Judd and Juillard to six decimals, which is asserted.

Copied on 2026-10-22 from the AI-ECON Session 4 lab (session04_ks.py).  Changes: the
network, the recording and the file-based warm start are left out (store_distribution
and warm_start_from replace the last); make_grid is ks_grid and solve_household is
solve_household_ks; the outer loop's bookkeeping moved into _pass_record; lines between
'# <<name>>' markers re-wrapped to fit a slide.  Imports: numpy only.
"""
from __future__ import annotations

import time
from dataclasses import dataclass, replace

import numpy as np

from s06_ha import lottery

# State order used everywhere, as in the published table:
#   0 = (recession, unemployed)  1 = (recession, employed)
#   2 = (boom, unemployed)       3 = (boom, employed)
Z_OF_S = np.array([0, 0, 1, 1])
E_OF_S = np.array([0, 1, 0, 1])

PUBLISHED_MATRIX = np.array([
    [0.525,    0.35,     0.03125,  0.09375],
    [0.038889, 0.836111, 0.002083, 0.122917],
    [0.09375,  0.03125,  0.291667, 0.583333],
    [0.009115, 0.115885, 0.024306, 0.850694]])

# Maliar, Maliar and Valli (JEDC 2010), non-stochastic simulation, same economy.
PUBLISHED_BENCHMARK = dict(
    source="Maliar, Maliar & Valli (2010), JEDC 34: law of motion under non-stochastic simulation",
    kappa_bad=[0.122146, 0.965942], kappa_good=[0.136272, 0.963582], K_mean=39.037,
    note="their stochastic-simulation solution: Den Haan error 0.050% mean, 0.156% max over 10,000 periods")


# -------------------------------------------------------------------------------------
# Calibration and numerical settings
# -------------------------------------------------------------------------------------
@dataclass(frozen=True)
class KSParams:
    """Den Haan, Judd and Juillard (2010), Table 1; quarterly."""
    beta: float = 0.99
    gamma: float = 1.0        # relative risk aversion; 1 is log utility
    alpha: float = 0.36
    delta: float = 0.025
    delta_a: float = 0.01     # productivity is 1 - delta_a in a recession, 1 + delta_a in a boom
    u_b: float = 0.10         # unemployment rate in a recession
    u_g: float = 0.04         # ... and in a boom
    mu: float = 0.15          # unemployment benefit as a fraction of the wage
    dur_z: float = 8.0        # expected duration of a boom or a recession, in quarters
    dur_u_b: float = 2.5      # expected duration of unemployment while the recession lasts
    dur_u_g: float = 1.5      # ... while the boom lasts
    a_min: float = 0.0        # k' >= 0

    @property
    def l_bar(self) -> float:             # time endowment: labour supply in a recession is one
        return 1.0 / (1.0 - self.u_b)

    @property
    def z(self) -> np.ndarray:
        return np.array([1.0 - self.delta_a, 1.0 + self.delta_a])

    @property
    def u(self) -> np.ndarray:
        return np.array([self.u_b, self.u_g])


@dataclass(frozen=True)
class KSSettings:
    n_a: int = 120            # household policy grid: nodes, top, curvature
    a_max: float = 300.0      # nothing may pile up at the top: the right tail is long
    curv: float = 2.5
    h_step: float = 0.1       # histogram grid: uniform on [0, a_max]; individual wealth is
                              # close to a random walk, so the histogram has to be fine
    n_K: int = 7              # aggregate capital: nodes, equally spaced in logs
    K_lo: float = 30.0
    K_hi: float = 50.0
    T_sim: int = 3000         # fitting sample, including the burn-in
    T_burn: int = 500
    T_oos: int = 2100         # fresh path for the accuracy test, including its burn-in
    T_oos_burn: int = 100
    damping: float = 0.3      # weight on the newly estimated coefficients
    tol_coef: float = 1e-5
    tol_egm: float = 1e-8
    max_outer: int = 80
    max_egm: int = 6000
    n_coarse: int = 20        # nodes of a coarse histogram (kept for the record)
    K0: float = 40.0          # everyone starts here; the burn-in removes it
    seed: int = 20260920
    seed_oos: int = 20261016


FULL = KSSettings()
SHORT = replace(FULL, T_sim=1100, T_burn=100, T_oos=1100, T_oos_burn=100, max_outer=3)


class KGridError(RuntimeError):
    """Aggregate capital, or its forecast, reached the edge of the K grid."""


def ks_grid(n, lo, hi, curv):
    return lo + (hi - lo) * np.linspace(0.0, 1.0, n) ** curv


# -------------------------------------------------------------------------------------
# The exogenous process
# -------------------------------------------------------------------------------------
def ks_transition_matrix(p: KSParams, check=True):
    """Joint chain over (aggregate state, employment), built from the published restrictions.

    Returns (Pz, P): the 2x2 chain for z and the 4x4 joint chain in the state order above.
    """
    stay = 1.0 - 1.0 / p.dur_z
    Pz = np.array([[stay, 1.0 - stay], [1.0 - stay, stay]])
    u, dur_u = p.u, (p.dur_u_b, p.dur_u_g)
    # probability of staying unemployed, conditional on the aggregate move z -> z'
    uu = np.empty((2, 2))
    uu[0, 0] = 1.0 - 1.0 / dur_u[0]
    uu[1, 1] = 1.0 - 1.0 / dur_u[1]
    uu[1, 0] = 1.25 * uu[0, 0]            # boom -> recession: harder to leave unemployment
    uu[0, 1] = 0.75 * uu[1, 1]            # recession -> boom: easier
    P = np.zeros((4, 4))
    for zi in range(2):
        for zj in range(2):
            # the employed lose jobs at the rate that makes next period's unemployment u(z')
            eu = (u[zj] - u[zi] * uu[zi, zj]) / (1.0 - u[zi])
            P[2 * zi + 0, 2 * zj + 0] = Pz[zi, zj] * uu[zi, zj]
            P[2 * zi + 0, 2 * zj + 1] = Pz[zi, zj] * (1.0 - uu[zi, zj])
            P[2 * zi + 1, 2 * zj + 0] = Pz[zi, zj] * eu
            P[2 * zi + 1, 2 * zj + 1] = Pz[zi, zj] * (1.0 - eu)
    if check:
        assert np.allclose(P.sum(axis=1), 1.0, atol=1e-12), "rows must sum to one"
        if p == KSParams():
            dev = float(np.abs(P - PUBLISHED_MATRIX).max())
            assert dev < 1e-6, f"transition matrix differs from the published table by {dev:.2e}"
    return Pz, P


def prices(K, zi, p: KSParams):
    """Rental rate, wage and tax rate at aggregate capital K in aggregate state zi."""
    L = 1.0 - p.u[zi]
    kl = K / (p.l_bar * L)
    r = p.alpha * p.z[zi] * kl ** (p.alpha - 1.0)
    w = (1.0 - p.alpha) * p.z[zi] * kl ** p.alpha
    tau = p.mu * p.u[zi] / (p.l_bar * L)          # balances the benefit budget every period
    return r, w, tau


def draw_z(T, Pz, rng, z0):
    z = np.empty(T, dtype=int)
    z[0] = z0
    draws = rng.random(T)
    for t in range(1, T):
        z[t] = z[t - 1] if draws[t] < Pz[z[t - 1], z[t - 1]] else 1 - z[t - 1]
    return z


# -------------------------------------------------------------------------------------
# The household, given a perceived law of motion  log K' = kappa[z,0] + kappa[z,1] log K
# -------------------------------------------------------------------------------------
def _resources(p: KSParams, K):
    """Gross return and non-asset income for each of the four states at capital K."""
    K = np.asarray(K, dtype=float)
    R = np.empty((4,) + K.shape)
    inc = np.empty((4,) + K.shape)
    for s in range(4):
        r, w, tau = prices(K, Z_OF_S[s], p)
        R[s] = 1.0 - p.delta + r
        inc[s] = w * ((1.0 - tau) * p.l_bar if E_OF_S[s] else p.mu)
    return R, inc


def check_rule_inside(kappa, logK):
    """The forecast must map the K grid into itself, or the household would read a
    clamped policy: stop and say so."""
    logKp = kappa[:, 0][:, None] + kappa[:, 1][:, None] * logK[None, :]
    if logKp.min() <= logK[0] or logKp.max() >= logK[-1]:
        raise KGridError(f"forecast K' in [{np.exp(logKp.min()):.2f}, "
                         f"{np.exp(logKp.max()):.2f}] reaches the K grid "
                         f"[{np.exp(logK[0]):.2f}, {np.exp(logK[-1]):.2f}]: widen it")
    return logKp


def _back_to_grid(a, x, y, R, inc, a_min):
    """Interpolate every column of the endogenous-grid solution (x, y) back to the fixed grid.

    Columns are laid end to end with an offset that keeps the abscissae increasing. Queries
    below a column's first node are where the constraint binds; queries above its last node
    continue linearly.
    """
    nC = x.shape[1]
    off = np.arange(nC) * (2.0 * (max(a[-1], x.max()) - min(a[0], x.min())) + 1.0)
    cc = np.interp((a[:, None] + off).T.ravel(), (x + off).T.ravel(),
                   y.T.ravel()).reshape(nC, a.size).T
    slope = (y[-1] - y[-2]) / (x[-1] - x[-2])
    cc = np.where(a[:, None] > x[-1], y[-1] + slope * (a[:, None] - x[-1]), cc)
    return np.where(a[:, None] < x[0], R * a[:, None] + inc - a_min, cc)


def solve_household_ks(kappa, p: KSParams, a, logK, P, c_init=None, tol=1e-8,
                       max_iter=6000):
    """Endogenous grid method. Returns (consumption c[a, s, K-node], sweeps, last change)."""
    nA, nK = a.size, logK.size
    R, inc = _resources(p, np.exp(logK))                         # (4, nK)
    logKp = check_rule_inside(kappa, logK)                       # (2, nK): log K' by z
    pos = (logKp - logK[0]) / (logK[1] - logK[0])
    m0 = np.clip(np.floor(pos).astype(int), 0, nK - 2)
    th = pos - m0
    Rp, _ = _resources(p, np.exp(logKp))                         # (4, 2, nK): R' by s'
    c = ((R - 1.0)[None] * a[:, None, None] + inc[None] if c_init is None
         else c_init.copy())
    c = np.maximum(c, 1e-10)
    Rf, incf = R.reshape(-1), inc.reshape(-1)
    diff, it = np.inf, 0
    for it in range(1, max_iter + 1):                          # one EGM sweep, kappa fixed
        cn = (1.0 - th) * c[:, :, m0] + th * c[:, :, m0 + 1]   # c'(a', s', K'): K' from the rule
        MU = Rp[None] * cn ** (-p.gamma)                       # R' u'(c')
        Euc = np.einsum("sp,apsm->asm", P, MU[:, :, Z_OF_S, :])  # E[R' u'(c') | s]
        c_end = (p.beta * Euc) ** (-1.0 / p.gamma)            # invert the Euler equation
        a_end = (c_end + a[:, None, None] - inc[None]) / R[None]   # endogenous grid
        c_new = _back_to_grid(a, a_end.reshape(nA, -1), c_end.reshape(nA, -1),
                              Rf, incf, p.a_min)               # limit: a comparison
        c_new = np.maximum(c_new.reshape(nA, 4, nK), 1e-10)
        diff = float(np.abs(c_new - c).max())
        c = c_new
        if diff < tol:
            break
    return c, it, diff


# -------------------------------------------------------------------------------------
# Non-stochastic simulation of the cross-section
# -------------------------------------------------------------------------------------
def hist_grid(s: KSSettings, a_min=0.0):
    n = int(round((s.a_max - a_min) / s.h_step)) + 1
    return np.linspace(a_min, s.a_max, n)


def coarse_projection(h, n_coarse, curv=2.0):
    """Mean-preserving projection of the fine histogram onto a few nodes: K = nodes @ x."""
    nodes = ks_grid(n_coarse, h[0], h[-1], curv)
    j, w_hi = lottery(h, nodes)
    W = np.zeros((h.size, n_coarse))
    W[np.arange(h.size), j] += 1.0 - w_hi
    W[np.arange(h.size), j + 1] += w_hi
    return nodes, W


def initial_distribution(h, K0, u0):
    j, w_hi = lottery(np.array([K0]), h)
    col = np.zeros(h.size)
    col[j[0]], col[j[0] + 1] = 1.0 - w_hi[0], w_hi[0]
    return np.column_stack([u0 * col, (1.0 - u0) * col])


def restart_distribution(lam, u0):
    """Re-use a distribution as a starting point in a period whose unemployment rate is u0."""
    out = lam.copy()
    out[:, 0] *= u0 / out[:, 0].sum()
    out[:, 1] *= (1.0 - u0) / out[:, 1].sum()
    return out


def simulate(c, p: KSParams, a, logK, Pz, P, z, lam0, h, W=None, burn=0):
    """Push the histogram over (wealth, employment) along the path z. No individuals are
    drawn: no sampling error.

    The policy lives on the grid `a`; the histogram lives on the fine uniform grid `h`, and
    Young's lottery puts each household's choice back on it without changing the mean.
    Returns K (one entry per period), the last distribution, the time-averaged cross-section
    after `burn`, the gate statistics and, if W is given, the coarse histogram each period.
    Raises KGridError if K touches the edge of the K grid.
    """
    T, nH, nK = z.size, h.size, logK.size
    dlog, K_lo, K_hi = logK[1] - logK[0], np.exp(logK[0]), np.exp(logK[-1])
    dh = h[1] - h[0]
    l_bar, mu, delta, alpha, u, zv = p.l_bar, p.mu, p.delta, p.alpha, p.u, p.z
    K = np.empty(T)
    X = np.empty((T, W.shape[1])) if W is not None else None
    mass_err = unemp_err = top = 0.0
    cross = np.zeros(nH)
    lam = lam0.copy()
    for t in range(T):
        marg = lam.sum(axis=1)
        Kt = float(h @ marg)
        if not (K_lo < Kt < K_hi):
            raise KGridError(f"K = {Kt:.3f} at t = {t} is outside the K grid "
                             f"({K_lo:.1f}, {K_hi:.1f}): widen it")
        K[t] = Kt
        if X is not None:
            X[t] = marg @ W
        zi = z[t]
        mass_err = max(mass_err, abs(marg.sum() - 1.0))
        unemp_err = max(unemp_err, abs(lam[:, 0].sum() - u[zi]))
        top = max(top, marg[-1])
        if t >= burn:
            cross += marg
        if t == T - 1:
            break
        pos = (np.log(Kt) - logK[0]) / dlog
        m0 = min(int(pos), nK - 2)
        th = pos - m0
        s0 = 2 * zi
        ct = (1.0 - th) * c[:, s0:s0 + 2, m0] + th * c[:, s0:s0 + 2, m0 + 1]  # (nA, 2)
        L = 1.0 - u[zi]
        kl = Kt / (l_bar * L)
        r = alpha * zv[zi] * kl ** (alpha - 1.0)
        w = (1.0 - alpha) * zv[zi] * kl ** alpha
        tau = mu * u[zi] / (l_bar * L)
        income = (mu * w, (1.0 - tau) * l_bar * w)
        moved = np.empty((nH, 2))
        for e in range(2):                           # a' on the histogram nodes h
            coh = (1.0 - delta + r) * h + income[e]  # r, w, income all from (K_t, z_t)
            ap = np.clip(coh - np.interp(h, a, ct[:, e]), p.a_min, h[-1])   # policy read at K_t
            q = (ap - h[0]) / dh                     # Young's lottery, uniform grid
            j = np.minimum(q.astype(np.int64), nH - 2)   # the lower node ...
            w_hi = q - j                             # ... and the share sent up
            moved[:, e] = (np.bincount(j, weights=lam[:, e] * (1.0 - w_hi), minlength=nH)
                           + np.bincount(j + 1, weights=lam[:, e] * w_hi, minlength=nH))
        zj = z[t + 1]                                # employment moves, given z -> z'
        lam = moved @ (P[s0:s0 + 2, 2 * zj:2 * zj + 2] / Pz[zi, zj])   # lambda_{t+1}
    return dict(K=K, z=z, X=X, lam_end=lam, cross_section=cross / max(1, T - burn),
                mass_error_max=mass_err, unemployment_error_max=unemp_err,
                mass_at_top_max=top)


def wealth_statistics(h, mass) -> dict:
    m = mass / mass.sum()
    cdf = np.cumsum(m)
    S = np.cumsum(m * h)
    Lz = np.concatenate([[0.0], S / S[-1]])
    F = np.concatenate([[0.0], cdf])
    pct = {f"p{q}": float(h[np.searchsorted(cdf, q / 100.0)]) for q in (10, 50, 90, 99)}
    return dict(gini=float(1.0 - np.sum((F[1:] - F[:-1]) * (Lz[1:] + Lz[:-1]))),
                mean=float(S[-1]), share_at_constraint=float(m[0]),
                share_below_one=float(m[h < 1.0].sum()),
                top_1pct_share=float(1.0 - np.interp(0.99, F, Lz)),
                maximum_with_mass=float(h[m > 1e-12].max()), **pct)


def fit_rule(K, z, start):
    """Least squares of log K' on log K, separately by today's aggregate state."""
    x, y, zz = np.log(K[start:-1]), np.log(K[start + 1:]), z[start:-1]   # after the burn-in
    kappa, r2 = np.empty((2, 2)), np.empty(2)
    for zi in range(2):                                        # recession, boom
        xi, yi = x[zz == zi], y[zz == zi]                      # the quarters with z_t = zi
        A = np.column_stack([np.ones_like(xi), xi])            # regressors: 1, log K_t
        b, *_ = np.linalg.lstsq(A, yi, rcond=None)             # OLS: (kappa_0, kappa_1)
        kappa[zi] = b
        r2[zi] = 1.0 - ((yi - A @ b) ** 2).sum() / ((yi - yi.mean()) ** 2).sum()  # R^2
    return kappa, r2


# -------------------------------------------------------------------------------------
# The algorithm: solve, simulate, regress, update
# -------------------------------------------------------------------------------------
def rule_gap_pct(kappa_a, kappa_b, K_lo, K_hi):
    """Largest difference between two rules AS FORECASTS, in per cent of K', over the range
    K actually visits.  Intercept and slope move against each other from one sample to the
    next, so coefficients can differ in the third decimal while the forecasts agree to a few
    parts in a hundred thousand.
    """
    d = np.asarray(kappa_a) - np.asarray(kappa_b)
    lk = np.log(np.linspace(K_lo, K_hi, 50))
    return float(100.0 * np.abs(d[:, :1] + d[:, 1:] * lk[None, :]).max())


def default_kappa():
    """A flat guess whose resting point sits in the middle of the K grid."""
    return np.array([[0.04 * np.log(38.5), 0.96], [0.04 * np.log(39.5), 0.96]])


def _say(verbose, text):
    if verbose:
        print(text)


def _pass_record(outer, it, change, kappa, kappa_hat, sim, r2, t1, verbose):
    """One row of the convergence table (and a printed line if verbose)."""
    gap = rule_gap_pct(kappa_hat, kappa, sim["K"].min(), sim["K"].max())
    row = dict(iteration=outer, egm_sweeps=it, change=change, forecast_gap_pct=gap,
               kappa0_bad=float(kappa_hat[0, 0]), kappa1_bad=float(kappa_hat[0, 1]),
               kappa0_good=float(kappa_hat[1, 0]), kappa1_good=float(kappa_hat[1, 1]),
               r2_bad=float(r2[0]), r2_good=float(r2[1]), seconds=time.time() - t1)
    _say(verbose, f"   pass {outer:2d}: {it:4d} household sweeps, coefficients moved "
                  f"{change:.2e}, forecasts moved {gap:.5f}%, R2 = {r2[0]:.6f} / "
                  f"{r2[1]:.6f}, {time.time() - t1:.1f} s")
    return row


def solve_ks(p: KSParams = KSParams(), s: KSSettings = FULL, kappa_init=None,
             lam_init=None, z0=1, verbose=True) -> dict:
    """Krusell-Smith fixed point in the coefficients of the perceived law of motion.

    kappa_init, lam_init, z0: a warm start (coefficients, a cross-section, and the
    aggregate state it belongs to).  Without one, everybody starts with K0 and the
    distribution found by each pass starts the next one.
    """
    t0 = time.time()
    Pz, P = ks_transition_matrix(p)
    a = ks_grid(s.n_a, p.a_min, s.a_max, s.curv)
    h = hist_grid(s, p.a_min)
    logK = np.linspace(np.log(s.K_lo), np.log(s.K_hi), s.n_K)
    nodes, W = coarse_projection(h, s.n_coarse)
    z = draw_z(s.T_sim, Pz, np.random.default_rng(s.seed), z0=z0)
    lam0 = (initial_distribution(h, s.K0, p.u[z0]) if lam_init is None
            else restart_distribution(lam_init, p.u[z0]))
    kappa = default_kappa() if kappa_init is None else np.array(kappa_init, dtype=float)
    c, rows, sweeps, widened = None, [], 0, 0
    sim = kappa_hat = r2 = None
    for outer in range(1, s.max_outer + 1):
        t1 = time.time()
        try:                                             # 1. household  2. simulate
            c, it, _ = solve_household_ks(kappa, p, a, logK, P, c_init=c,
                                          tol=s.tol_egm, max_iter=s.max_egm)
            sim = simulate(c, p, a, logK, Pz, P, z, lam0, h)
        except KGridError as err:                        # never clamp: widen, restart
            widened += 1
            if widened > 3:
                raise
            lo, hi = np.exp(logK[0]) * 0.85, np.exp(logK[-1]) * 1.15
            logK, c = np.linspace(np.log(lo), np.log(hi), s.n_K), None
            _say(verbose, f"   {err}\n   K grid widened to [{lo:.1f}, {hi:.1f}]")
            continue
        sweeps += it
        kappa_hat, r2 = fit_rule(sim["K"], z, s.T_burn)  # 3. regress, state by state
        change = float(np.abs(kappa_hat - kappa).max())
        rows.append(_pass_record(outer, it, change, kappa, kappa_hat, sim, r2, t1,
                                 verbose))
        if change < s.tol_coef:
            break
        kappa = (1.0 - s.damping) * kappa + s.damping * kappa_hat     # 4. damped update
        lam0 = restart_distribution(sim["lam_end"], p.u[z0])
    # `kappa` is the rule households used; `kappa_hat` is what their behaviour implies.
    sim = simulate(c, p, a, logK, Pz, P, z, lam0, h, W=W, burn=s.T_burn)
    return dict(params=p, settings=s, Pz=Pz, P=P, a=a, h=h, logK=logK, nodes=nodes, W=W,
                c=c, kappa=kappa, kappa_regression=kappa_hat, r2=r2, rows=rows, sim=sim,
                egm_sweeps=sweeps, converged=bool(rows and rows[-1]["change"] < s.tol_coef),
                K_grid_widened=widened, seconds=time.time() - t0)


# -------------------------------------------------------------------------------------
# Is the rule any good?  R^2 is a one-step statistic; Den Haan's test is not.
# -------------------------------------------------------------------------------------
def rule_path(kappa, K_start, z):
    """Iterate the perceived law forward from one K, never looking at the simulation."""
    Kf = np.empty(z.size)
    Kf[0] = K_start                                            # start at the true K, once
    for t in range(z.size - 1):                                # then forecast from the forecast
        Kf[t + 1] = np.exp(kappa[z[t], 0] + kappa[z[t], 1] * np.log(Kf[t]))   # z_t's row of kappa
    return Kf


def pct_log_error(K_hat, K):
    return 100.0 * np.abs(np.log(K_hat) - np.log(K))


def _summary(err):
    return dict(mean_pct=float(err.mean()), max_pct=float(err.max()), periods=int(err.size))


def diagnose_ks(sol, long_run=200000, verbose=True) -> dict:
    """In-sample and out-of-sample accuracy of the converged rule."""
    p, s, kappa = sol["params"], sol["settings"], sol["kappa"]
    fit = sol["sim"]
    b = s.T_burn
    err_in = pct_log_error(rule_path(kappa, fit["K"][b], fit["z"][b:]), fit["K"][b:])
    # a fresh shock path, started from where the fitting sample ended
    z_oos = draw_z(s.T_oos, sol["Pz"], np.random.default_rng(s.seed_oos), z0=int(fit["z"][-1]))
    oos = simulate(sol["c"], p, sol["a"], sol["logK"], sol["Pz"], sol["P"], z_oos,
                   fit["lam_end"], sol["h"], W=sol["W"], burn=s.T_oos_burn)
    bo = s.T_oos_burn
    K_rule = rule_path(kappa, oos["K"][bo], z_oos[bo:])
    err_out = pct_log_error(K_rule, oos["K"][bo:])
    one = np.exp(kappa[z_oos[bo:-1], 0] + kappa[z_oos[bo:-1], 1] * np.log(oos["K"][bo:-1]))
    err_one = pct_log_error(one, oos["K"][bo + 1:])
    z_long = draw_z(long_run, sol["Pz"], np.random.default_rng(s.seed_oos + 1), z0=1)
    K_long = rule_path(kappa, fit["K"][b:].mean(), z_long)[1000:]
    out = dict(r2=dict(bad=float(sol["r2"][0]), good=float(sol["r2"][1])),
               den_haan_in_sample=_summary(err_in), den_haan_out_of_sample=_summary(err_out),
               one_step_out_of_sample=_summary(err_one), oos=oos, K_rule=K_rule,
               err_out=err_out, err_in=err_in,
               wealth=wealth_statistics(sol["h"], fit["cross_section"]),
               K=dict(mean=float(fit["K"][b:].mean()), sd=float(fit["K"][b:].std()),
                      min=float(min(fit["K"].min(), oos["K"].min())),
                      max=float(max(fit["K"].max(), oos["K"].max())),
                      grid_lo=float(np.exp(sol["logK"][0])), grid_hi=float(np.exp(sol["logK"][-1])),
                      share_of_boom_quarters=float(fit["z"][b:].mean()),
                      rule_long_run_mean=float(K_long.mean()), rule_long_run_sd=float(K_long.std()),
                      rule_rest_bad=float(np.exp(kappa[0, 0] / (1.0 - kappa[0, 1]))),
                      rule_rest_good=float(np.exp(kappa[1, 0] / (1.0 - kappa[1, 1])))))
    if verbose:
        print(f"   R2 by state  {out['r2']['bad']:.7f} (recession)   {out['r2']['good']:.7f} (boom)")
        for key, label in (("den_haan_in_sample", "Den Haan error, fitting path   "),
                           ("den_haan_out_of_sample", "Den Haan error, FRESH path     "),
                           ("one_step_out_of_sample", "one-step error, fresh path     ")):
            d = out[key]
            print(f"   {label} mean {d['mean_pct']:.4f}%   max {d['max_pct']:.4f}%   ({d['periods']} quarters)")
    return out


def gates(sol, diag) -> dict:
    """The structural checks. Every entry is a number; `passed` collects the verdicts."""
    fit, oos, K = sol["sim"], diag["oos"], diag["K"]
    g = dict(matrix_max_abs_dev=float(np.abs(sol["P"] - PUBLISHED_MATRIX).max()),
             K_min=K["min"], K_max=K["max"], K_grid=(K["grid_lo"], K["grid_hi"]),
             K_grid_widened=int(sol["K_grid_widened"]),
             mass_error_max=float(max(fit["mass_error_max"], oos["mass_error_max"])),
             unemployment_error_max=float(max(fit["unemployment_error_max"],
                                              oos["unemployment_error_max"])),
             mass_at_top_max=float(max(fit["mass_at_top_max"], oos["mass_at_top_max"])),
             share_at_constraint=float(diag["wealth"]["share_at_constraint"]),
             coefficient_change=float(sol["rows"][-1]["change"]),
             rule_gap_pct=float(sol["rows"][-1]["forecast_gap_pct"]))
    g["passed"] = dict(matrix=g["matrix_max_abs_dev"] < 1e-6,
                       K_strictly_inside_grid=K["grid_lo"] < K["min"] and K["max"] < K["grid_hi"],
                       mass_is_one=g["mass_error_max"] < 1e-10,
                       unemployment_matches=g["unemployment_error_max"] < 1e-10,
                       # below 1e-6 the ceiling cannot move K by one part in a hundred thousand
                       nothing_at_the_top=g["mass_at_top_max"] < 1e-6,
                       # the rule households use and the rule their behaviour implies agree
                       rule_is_a_fixed_point=g["rule_gap_pct"] < 0.005)
    return g


# -------------------------------------------------------------------------------------
# Warm starts: a cross-section kept on a coarser grid, and put back on a run's histogram
# -------------------------------------------------------------------------------------
def store_distribution(sol, step=0.5) -> dict:
    """The cross-section at the end of the fitting sample, on a coarser grid that keeps
    its mean, for warm starts."""
    h, lam = sol["h"], sol["sim"]["lam_end"]
    grid = np.arange(h[0], h[-1] + 0.5 * step, step)
    j, w_hi = lottery(h, grid)
    out = np.zeros((grid.size, 2))
    for e in range(2):
        out[:, e] = (np.bincount(j, weights=lam[:, e] * (1.0 - w_hi), minlength=grid.size)
                     + np.bincount(j + 1, weights=lam[:, e] * w_hi, minlength=grid.size))
    last = int(np.nonzero(out.sum(axis=1) > 1e-13)[0].max()) + 1
    return dict(note="end of the fitting sample; mass at wealth = step * index, by employment",
                step=step, z=int(sol["sim"]["z"][-1]),
                unemployed=[float(f"{v:.9e}") for v in out[:last, 0]],
                employed=[float(f"{v:.9e}") for v in out[:last, 1]])


def warm_start_from(ws, kappa, s: KSSettings, a_min=0.0):
    """Coefficients, cross-section and aggregate state from a stored run, put on this
    run's histogram grid."""
    h = hist_grid(s, a_min)
    lam = np.zeros((h.size, 2))
    where = np.clip(np.round((np.arange(len(ws["unemployed"])) * ws["step"] - h[0])
                             / s.h_step).astype(int), 0, h.size - 1)
    np.add.at(lam[:, 0], where, ws["unemployed"])
    np.add.at(lam[:, 1], where, ws["employed"])
    return np.array(kappa, dtype=float), lam, int(ws["z"])

In [ ]:
%%writefile l08_code/s08_ha_ac.py
"""Session 8 -- the Aiyagari household's actor-critic with the histogram as a state (8.B1, Stage 1B; L08 Part B).

Written for this course on 2026-11-11 from 8.B1's frames ("The Problem the Two Networks Solve" to "The
Algorithm"), at CPU scale and on Session 6's annual Aiyagari economy (s06_ha.Params: alpha .33, beta .96,
delta .08, CRRA 2, a seven-state Rouwenhorst chain, no labour choice, no government). The method is
Feng, Han, Sargent and Zhu's Stage 1B (research in progress); this file is not their code and carries none of their numbers.

The state of a household is (eps, a; Gamma): its productivity, its assets and the whole distribution Gamma, a
histogram of masses Gamma[i, q] on the asset nodes abar_i (the histogram grid) and the productivity levels eps_q.
  prices      K(Gamma) = sum_iq Gamma[i, q] abar_i, L = 1; r = alpha K^(alpha - 1) - delta, w = (1 - alpha) K^alpha
  actor       a' = g_theta(eps, a; Gamma) = s cash, s = min(max(s*(eps, a) + f_theta, 0), 0.99): a saving share,
              s* 6.A1's stationary EGM rule as a share of cash at r* (interpolated in a), f_theta a correction
              that starts at zero; a' >= 0 built in
  law         Gamma_+ = Gamma P(Gamma; theta): 6.A2's Young lottery on the actor's a' at every node, then Pi
  critic      V_psi(eps, a; Gamma), fitted to b = u(c) + beta sum_q' Pi[q, q'] V_psi-(eps_q', a'; Gamma_+),
              with psi- a trailing copy (psi- <- (1 - tau) psi- + tau psi)
  actor step  maximise J = mean[u(c) + beta C(a')], C(a') = sum_q' Pi[q, q'] V_psi-(eps_q', a'; Gamma_+), whose
              slope in a' is replaced by the envelope condition beta sum_q' Pi[q, q'] (1 + r_+) u'(c_+), so the
              gradient is the Euler residual; a hinge barrier keeps c > 0; a rollout penalty
              nu_K mean_t (K_t / K* - 1)^2 over T_roll periods from lambda*, inside the graph, pins K
  batch       E tilted distributions Gamma ~ lambda* exp(xi abar / K*), xi uniform on [-xi_max, xi_max], each
              productivity column rescaled so that its mass stays pi_q
  loop        a critic pre-fit to the starting actor, then per iteration n_critic critic steps and n_actor actor
              steps on a fresh batch
Answer keys (Session 6 code, on the same histogram grid): the stationary equilibrium (6.A2's bisection) gives
lambda*, K*, r* and the EGM rule; 6.B2's perfect-foresight loop from a tilted Gamma_0 gives the equilibrium path
of K_t, which is what a Markov rule g(eps, a; Gamma) must reproduce when there are no aggregate shocks.

Departures from 8.B1, each said in L08: Session 6's calibration (no labour, so y = c and no psi_l, theta_l);
N = 40 asset nodes, not 140; the actor is the one stationary EGM rule plus a learned correction, in place of a
warm start from one EGM rule per sampled K (at a sampled K below K* the constant-price problem has r > 1/beta - 1
and no stationary solution); the
networks read the joint histogram (N x 7 masses) through a first layer split into a node part and a
distribution part, which is the same dense layer on the concatenated input, computed once per economy.
Imports: numpy, torch, s06_ha, s06_paths.
"""
from __future__ import annotations

import copy
import time
from dataclasses import dataclass, asdict, replace

import numpy as np
import torch
import torch.nn as nn

from s06_ha import Params, solve_equilibrium, lottery, push_forward
from s06_paths import backward_policies, forward_capital, path_prices, steady_state_value

torch.set_num_threads(int(__import__('os').environ.get('S08_THREADS', 4)))
DT = torch.float64


# ---------------------------------------------------------------------------------------------- settings
@dataclass(frozen=True)
class Config:
    n_bins: int = 40          # asset nodes of the histogram (8.B1: 140)
    a_max: float = 60.0       # the top node abar_N
    curv: float = 2.5         # node spacing, dense near the limit (6.A1's grid)
    E: int = 16               # tilted distributions per iteration
    xi_max: float = 0.2       # tilt range: Gamma ~ lambda* exp(xi abar / K*); K from 0.84 to 1.24 K*
    hidden: int = 64          # two tanh layers in each network
    iters: int = 400          # iterations of the loop
    n_critic: int = 8         # critic steps per iteration (8.B1: 8)
    n_actor: int = 2          # actor steps per iteration (8.B1: 2)
    lr_critic: float = 1e-3
    lr_actor: float = 1e-4
    tau: float = 0.02         # the trailing copy's nudge (8.B1: 0.02)
    prefit: int = 600         # critic steps before the first actor step
    T_roll: int = 20          # rollout periods inside the actor's loss (8.B1: 100)
    nu_K: float = 10.0        # weight of the rollout penalty
    c_floor: float = 1e-4     # the barrier's threshold
    kappa: float = 1.0        # the barrier's weight
    log_every: int = 50
    seed: int = 0


LIVE = Config()
LAB = replace(LIVE, E=8, T_roll=10, iters=150, prefit=300)       # the notebook's live run, about 1.5 minutes
SMOKE = replace(LIVE, E=4, iters=6, prefit=10, T_roll=5, log_every=2)


# ---------------------------------------------------------------------------------------------- answer key
def answer_key(cfg: Config) -> dict:
    """Session 6's stationary equilibrium on the histogram grid: lambda*, K*, r*, the EGM rule and its value."""
    p = Params(n_a=cfg.n_bins, a_max=cfg.a_max, curv=cfg.curv)
    sol = solve_equilibrium(p)                                 # 6.A2: bisection on r, warm-started
    V = steady_state_value(sol["c"], sol["ap"], sol["grid"], sol["Pi"], p)   # value of the rule for ever
    sol.update(V=V, K_star=float(sol["K_supply"]), params=p)
    return sol


class Economy:
    """The tensors every step needs: nodes, levels, Pi, the answer key, and the value scale."""

    def __init__(self, key: dict):
        self.p = key["params"]
        self.grid = torch.as_tensor(key["grid"], dtype=DT)     # abar_i, (N,)
        self.e = torch.as_tensor(key["e"], dtype=DT)           # eps_q, (Q,)
        self.Pi = torch.as_tensor(key["Pi"], dtype=DT)         # Pi[q, q'], (Q, Q)
        self.pi = torch.as_tensor(key["pi"], dtype=DT)         # invariant shares of eps, (Q,)
        self.lam = torch.as_tensor(key["lam"], dtype=DT)       # lambda*, (N, Q)
        self.ap = torch.as_tensor(key["ap"], dtype=DT)         # the stationary EGM rule, (N, Q)
        self.K_star = float(key["K_star"])
        self.r_star, self.w_star = float(key["r"]), float(key["w"])
        self.N, self.Q = self.ap.shape
        V = torch.as_tensor(key["V"], dtype=DT)
        self.V_mu, self.V_sd = float(V.mean()), float(V.std())   # the critic's level and scale

    def u(self, c):
        s = self.p.sigma
        return torch.log(c) if s == 1.0 else c ** (1.0 - s) / (1.0 - s)

    def up(self, c):                                           # u'(c)
        return c ** (-self.p.sigma)

    def prices(self, Gamma):
        """Gamma (E, N, Q) -> K, r, w, each (E,). L = 1 because every tilt keeps the eps-marginal at pi."""
        K = (Gamma.sum(-1) * self.grid).sum(-1)
        r = self.p.alpha * K ** (self.p.alpha - 1.0) - self.p.delta
        w = (1.0 - self.p.alpha) * K ** self.p.alpha
        return K, r, w

    def cash(self, a, q_level, r, w):
        """Cash on hand (1 + r) a + w eps; a, q_level (E, ...), r, w (E,)."""
        shape = (-1,) + (1,) * (a.dim() - 1)
        return (1.0 + r.view(shape)) * a + w.view(shape) * q_level


# ---------------------------------------------------------------------------------------------- the networks
class DistNet(nn.Module):
    """f(eps, a; Gamma): two tanh layers. Layer 1 is W [x; d] + b, split as W_x x + (W_G d + b), so the
    distribution part is computed once per economy and broadcast to its nodes. Inputs are put on order one:
    x = (assets on a log scale in [-1, 1], log eps / its largest value); d = the histogram's relative deviation
    from lambda*, (Gamma - lambda*) / (lambda* + 1 / (N Q)), one entry per node."""

    def __init__(self, eco, hidden):
        super().__init__()
        self.node = nn.Linear(2, hidden, bias=False)
        self.dist = nn.Linear(eco.N * eco.Q, hidden)
        self.out = nn.Sequential(nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.register_buffer("lam", eco.lam.clone())
        self.log_top = float(np.log1p(float(eco.grid[-1])))
        self.le_top = float(torch.log(eco.e).abs().max())
        self.to(DT)

    def forward(self, a, logeps, Gamma):
        """a, logeps: (E, M) query points; Gamma: (E, N, Q). Returns (E, M)."""
        x = torch.stack([2.0 * torch.log1p(a) / self.log_top - 1.0, logeps / self.le_top], -1)   # (E, M, 2)
        d = (Gamma - self.lam) / (self.lam + 1.0 / self.lam.numel())                          # (E, N, Q)
        return self.out(self.node(x) + self.dist(d.flatten(1))[:, None, :]).squeeze(-1)


class Actor(nn.Module):
    """a' = g_theta(eps, a; Gamma) = s cash, s = min(max(s* + f, 0), S_MAX): a saving share. s*(eps, a) is 6.A1's
    stationary EGM rule as a share of cash on hand at r*, linear in a between the nodes; f is the network, its last
    layer zero at the start, so the first actor IS the stationary rule and the loop learns how Gamma moves it.
    max(., 0) is 8.B1's limit built into the rule (a' = 0 exactly where it binds, with zero slope there);
    S_MAX < 1 keeps c > 0 at every node, so u(c) is always defined; a' is also kept below the top node."""

    S_MAX = 0.99

    def __init__(self, eco: Economy, hidden):
        super().__init__()
        self.eco, self.f = eco, DistNet(eco, hidden)
        nn.init.zeros_(self.f.out[-1].weight); nn.init.zeros_(self.f.out[-1].bias)   # f = 0: the stationary rule
        cash_star = (1.0 + eco.r_star) * eco.grid[:, None] + eco.w_star * eco.e[None, :]
        self.register_buffer("s_star", (eco.ap / cash_star).T.contiguous())          # (Q, N): s* on the nodes
        self.register_buffer("le", torch.log(eco.e))

    def base_share(self, a, logeps):
        """s*(eps, a): which eps_q the query is, then linear interpolation of s* in a between the nodes."""
        g = self.eco.grid
        q = torch.argmin((logeps[..., None] - self.le).abs(), -1)                   # (E, M), an index
        a1 = torch.clamp(a, g[0], g[-1])
        j = torch.clamp(torch.searchsorted(g, a1.detach().contiguous(), right=True) - 1, 0, g.numel() - 2)
        t = (a1 - g[j]) / (g[j + 1] - g[j])
        s = self.s_star[q, j], self.s_star[q, j + 1]
        return (1.0 - t) * s[0] + t * s[1]

    def share_raw(self, a, logeps, Gamma):                            # s* + f, before the clamp
        return self.base_share(a, logeps) + self.f(a, logeps, Gamma)

    def forward(self, a, logeps, Gamma):
        _, r, w = self.eco.prices(Gamma)
        cash = self.eco.cash(a, torch.exp(logeps), r, w)                 # (1 + r) a + w eps
        s = torch.clamp(self.share_raw(a, logeps, Gamma), 0.0, self.S_MAX)
        return torch.minimum(s * cash, self.eco.grid[-1])


class Critic(nn.Module):
    """V_psi(eps, a; Gamma) = V_mu + V_sd net: the answer key's level and scale, so the net is order one."""

    def __init__(self, eco: Economy, hidden):
        super().__init__()
        self.eco, self.f = eco, DistNet(eco, hidden)

    def forward(self, a, logeps, Gamma):
        return self.eco.V_mu + self.eco.V_sd * self.f(a, logeps, Gamma)


def polyak(target, net, tau):
    """psi- <- (1 - tau) psi- + tau psi: the trailing copy the critic's target is read from."""
    with torch.no_grad():
        for pt, p in zip(target.parameters(), net.parameters()):
            pt.mul_(1.0 - tau).add_(p, alpha=tau)


# ---------------------------------------------------------------------------------------------- the law of motion
def node_queries(eco: Economy, E):
    """Every node (abar_i, eps_q) of every economy, flattened to M = N Q query points: a, log eps, eps."""
    a = eco.grid[:, None].expand(eco.N, eco.Q).reshape(1, -1).expand(E, -1)
    eps = eco.e[None, :].expand(eco.N, eco.Q).reshape(1, -1).expand(E, -1)
    return a, torch.log(eps), eps


def lottery_batch(ap, grid):
    """6.A2's Young lottery in torch, batched: the node index j (no gradient) and the weight sent up (gradient)."""
    a1 = torch.clamp(ap, grid[0], grid[-1])
    j = torch.clamp(torch.searchsorted(grid, a1.detach().contiguous(), right=True) - 1, 0, grid.numel() - 2)
    return j, (a1 - grid[j]) / (grid[j + 1] - grid[j])


def push_batch(Gamma, ap, eco: Economy):
    """Gamma_+ = Gamma P(Gamma; theta): split each node's mass between the neighbours of a', then eps -> eps'.
    Gamma, ap: (E, N, Q). Differentiable in ap through the lottery weights."""
    E, N, Q = Gamma.shape
    j, w_hi = lottery_batch(ap, eco.grid)
    lo = torch.zeros(E, N, Q, dtype=DT).scatter_add(1, j, Gamma * (1.0 - w_hi))   # mass kept at a_j ...
    hi = torch.zeros(E, N, Q, dtype=DT).scatter_add(1, j + 1, Gamma * w_hi)       # ... and sent to a_j+1
    return torch.einsum("enq,qp->enp", lo + hi, eco.Pi)                          # then Pi[q, q']


def policy_on_nodes(actor, eco: Economy, Gamma):
    """The actor's a' at every node of every Gamma: (E, N, Q)."""
    a, le, _ = node_queries(eco, Gamma.shape[0])
    return actor(a, le, Gamma).view(-1, eco.N, eco.Q)


def tilted_batch(eco: Economy, xi):
    """Gamma ~ lambda* exp(xi abar / K*), each eps column rescaled to keep its mass pi_q. xi: (E,)."""
    w = torch.exp(xi[:, None] * eco.grid[None, :] / eco.K_star)                  # (E, N)
    G = eco.lam[None] * w[:, :, None]
    return G * (eco.pi / G.sum(1))[:, None, :]


# ---------------------------------------------------------------------------------------------- the two steps
def bellman_pieces(actor, target, eco: Economy, Gamma, cfg: Config, grad_actor: bool):
    """At every node of every Gamma: c, u(c), the barrier, C(a') from the trailing critic, and the envelope
    slope's surrogate. Gamma_+ is the household's given (no gradient)."""
    E = Gamma.shape[0]
    a, le, eps = node_queries(eco, E)
    K, r, w = eco.prices(Gamma)
    ap = actor(a, le, Gamma) if grad_actor else actor(a, le, Gamma).detach()     # (E, M)
    c = eco.cash(a, eps, r, w) - ap
    U = eco.u(torch.clamp(c, min=cfg.c_floor))
    B = torch.relu(cfg.c_floor - c) / cfg.c_floor                                # zero where c is feasible
    with torch.no_grad():
        Gp = push_batch(Gamma, ap.detach().view(E, eco.N, eco.Q), eco)           # Gamma_+, given
        _, rp, wp = eco.prices(Gp)
    # tomorrow at each q': rows (E, M, Q), the same a' for every q'
    apq = ap[:, :, None].expand(-1, -1, eco.Q)
    leq = torch.log(eco.e)[None, None, :].expand_as(apq)
    epq = eco.e[None, None, :].expand_as(apq)
    probs = eco.Pi[None].expand(eco.N, -1, -1).reshape(1, eco.N * eco.Q, eco.Q)  # Pi[q, q'] by node
    with torch.no_grad():
        Vn = target(apq.detach().reshape(E, -1), leq.reshape(E, -1), Gp).view(E, -1, eco.Q)
        C = (probs * Vn).sum(-1)                                                 # sum_q' Pi V_psi-(eps', a'; G+)
        app = actor(apq.detach().reshape(E, -1), leq.reshape(E, -1), Gp).view(E, -1, eco.Q)
        cp = torch.clamp(eco.cash(apq.detach(), epq, rp, wp) - app, min=cfg.c_floor)
        slope = (probs * (1.0 + rp.view(-1, 1, 1)) * eco.up(cp)).sum(-1)          # envelope: dV/da' at a'
    S = slope * ap                                                               # carries the exact slope
    C_eff = C + (S - S.detach())                                                 # C's value, the envelope's slope
    return dict(ap=ap, c=c, U=U, B=B, C=C, C_eff=C_eff, Gp=Gp, slope=slope, r=r)


def critic_step(critic, target, actor, eco, Gamma, opt, cfg):
    """L_V = mean (V_psi - b)^2, b = U + beta sum_q' Pi V_psi-(eps', a'; Gamma_+) held fixed; then the nudge."""
    with torch.no_grad():
        pc = bellman_pieces(actor, target, eco, Gamma, cfg, grad_actor=False)
        b = pc["U"] + eco.p.beta * pc["C"]                                       # the label, a number
    a, le, _ = node_queries(eco, Gamma.shape[0])
    loss = torch.mean(((critic(a, le, Gamma) - b) / eco.V_sd) ** 2)              # in the critic's units
    opt.zero_grad(); loss.backward(); opt.step()
    polyak(target, critic, cfg.tau)                                              # psi- moves a little
    return loss.item()


def rollout_penalty(actor, eco: Economy, cfg: Config):
    """(1 / T) sum_t (K_t / K* - 1)^2 along Gamma_{t+1} = Gamma_t P(Gamma_t; theta) from lambda*, in the graph."""
    G = eco.lam[None]
    pen = 0.0
    for _ in range(cfg.T_roll):
        G = push_batch(G, policy_on_nodes(actor, eco, G), eco)                  # the lottery carries the gradient
        K = (G.sum(-1) * eco.grid).sum(-1)
        pen = pen + (K / eco.K_star - 1.0) ** 2
    return (pen / cfg.T_roll).mean()


def actor_step(actor, target, eco, Gamma, opt, cfg):
    """L_g = -J + kappa mean B + nu_K rollout: J's gradient is the Euler residual times d a'/d theta."""
    pc = bellman_pieces(actor, target, eco, Gamma, cfg, grad_actor=True)
    J = torch.mean(pc["U"] + eco.p.beta * pc["C_eff"])                           # the Bellman right-hand side
    roll = rollout_penalty(actor, eco, cfg) if cfg.nu_K > 0 else torch.zeros((), dtype=DT)
    loss = -J + cfg.kappa * pc["B"].mean() + cfg.nu_K * roll
    opt.zero_grad(); loss.backward(); opt.step()
    return J.item(), roll.item()


# ---------------------------------------------------------------------------------------------- the loop
def train(cfg: Config = LIVE, key: dict | None = None, verbose: bool = True):
    """Pre-fit the critic to the stationary rule, then iterations of n_critic critic and n_actor actor steps."""
    t0 = time.time()
    torch.manual_seed(cfg.seed)
    gen = torch.Generator().manual_seed(cfg.seed + 1)
    key = answer_key(cfg) if key is None else key
    eco = Economy(key)
    actor, critic = Actor(eco, cfg.hidden), Critic(eco, cfg.hidden)
    target = copy.deepcopy(critic)                                   # psi-, the trailing copy
    hist = dict(critic=[], J=[], roll=[], grade=[dict(iteration=-1, **policy_grade(actor, eco))])
    opt_c = torch.optim.Adam(critic.parameters(), lr=cfg.lr_critic)
    opt_a = torch.optim.Adam(actor.parameters(), lr=cfg.lr_actor)

    def batch():                                                     # E fresh tilted distributions
        xi = (2.0 * torch.rand(cfg.E, generator=gen, dtype=DT) - 1.0) * cfg.xi_max
        return tilted_batch(eco, xi)

    for _ in range(cfg.prefit):                                      # the critic learns this actor first
        hist["critic"].append(critic_step(critic, target, actor, eco, batch(), opt_c, cfg))
    for it in range(cfg.iters):
        G = batch()
        for _ in range(cfg.n_critic):
            lc = critic_step(critic, target, actor, eco, G, opt_c, cfg)
        for _ in range(cfg.n_actor):
            J, roll = actor_step(actor, target, eco, G, opt_a, cfg)
        hist["critic"].append(lc); hist["J"].append(J); hist["roll"].append(roll)
        if it % cfg.log_every == 0 or it == cfg.iters - 1:
            g = policy_grade(actor, eco)
            hist["grade"].append(dict(iteration=it, **g))
            if verbose:
                print(f"iter {it:4d} | critic {lc:.2e} | J {J:9.4f} | rollout {roll:.2e} | "
                      f"a' gap {g['ap_gap_mean']:.4f} | Euler {g['euler_mean']:.2e} | {time.time() - t0:5.0f} s", flush=True)
    hist["seconds"] = time.time() - t0
    hist["cfg"] = asdict(cfg)
    return dict(actor=actor, critic=critic, target=target, eco=eco, key=key, hist=hist)


# ---------------------------------------------------------------------------------------------- grades
@torch.no_grad()
def policy_grade(actor, eco: Economy) -> dict:
    """At Gamma = lambda*: the actor's a' against the stationary EGM rule (lambda*-weighted, in units of K*),
    and the Euler residual |1 - c_Euler / c| where the EGM rule is interior (a' > 0), c_Euler = (beta sum Pi (1 + r_+) u'(c_+))^(-1/sigma)."""
    G = eco.lam[None]
    ap = policy_on_nodes(actor, eco, G)[0]
    m = eco.lam / eco.lam.sum()
    gap = (ap - eco.ap).abs() / eco.K_star
    K, r, w = eco.prices(G)
    c = eco.cash(eco.grid[None, :, None].expand(1, eco.N, eco.Q), eco.e[None, None, :], r, w)[0] - ap
    Gp = push_batch(G, ap[None], eco)
    _, rp, wp = eco.prices(Gp)
    rows = ap[:, :, None].expand(-1, -1, eco.Q)
    le = torch.log(eco.e)[None, None, :].expand_as(rows)
    app = actor(rows.reshape(1, -1), le.reshape(1, -1), Gp).view(eco.N, eco.Q, eco.Q)
    cp = (1.0 + rp) * rows + wp * eco.e[None, None, :] - app
    rhs = eco.p.beta * (eco.Pi[None] * (1.0 + rp) * cp.clamp(min=1e-10) ** (-eco.p.sigma)).sum(-1)
    resid = (1.0 - rhs ** (-1.0 / eco.p.sigma) / c.clamp(min=1e-10)).abs()
    interior = (eco.ap > 1e-8) & (ap < eco.grid[-1] - 1e-6)                     # where the answer key's rule is
    mi = m * interior
    return dict(ap_gap_mean=float((m * gap).sum()), ap_gap_max=float(gap[eco.lam > 1e-8].max()),
                euler_mean=float((mi * resid).sum() / mi.sum()), share_interior=float(mi.sum()))


@torch.no_grad()
def actor_path(actor, eco: Economy, G0, T):
    """K_t under the actor's own law of motion, Gamma_{t+1} = Gamma_t P(Gamma_t; theta), from G0 (N, Q)."""
    G = torch.as_tensor(G0, dtype=DT)[None]
    K = [float((G.sum(-1) * eco.grid).sum())]
    for _ in range(T):
        G = push_batch(G, policy_on_nodes(actor, eco, G), eco)
        K.append(float((G.sum(-1) * eco.grid).sum()))
    return np.array(K), G[0].numpy()


def fixed_rule_path(eco: Economy, key: dict, G0, T):
    """K_t if every household kept 6.A1's stationary rule at r* whatever the prices: the rule that ignores Gamma."""
    lam = np.asarray(G0, dtype=float)
    K = [float((lam.sum(1) * key["grid"]).sum())]
    j, w_hi = lottery(key["ap"], key["grid"])
    for _ in range(T):
        lam = push_forward(lam, j, w_hi, key["Pi"])
        K.append(float((lam.sum(1) * key["grid"]).sum()))
    return np.array(K)


def perfect_foresight_path(key: dict, G0, T=150, damping=0.5, tol=1e-7, max_outer=400):
    """6.B2's transition loop with no tax, from Gamma_0 = G0 back to lambda*: guess K_t, prices, households
    backward from the stationary rule, the distribution forward from G0, damped update. The answer key."""
    p, grid, e, Pi = key["params"], key["grid"], key["e"], key["Pi"]
    lam0 = np.asarray(G0, dtype=float)
    K0 = float((lam0.sum(1) * grid).sum())
    Ks_star = key["K_star"]
    K = Ks_star + (K0 - Ks_star) * np.exp(-np.arange(T + 1) / 10.0)
    gap = np.inf
    for outer in range(1, max_outer + 1):
        r, r_net, w, transfer = path_prices(K, p, 0.0)
        c_path, ap_path = backward_policies(r_net, w, transfer, key["c"], grid, e, Pi, p)
        Ks, _, _ = forward_capital(lam0, ap_path, grid, Pi, K0)
        gap = float(np.abs(Ks - K).max() / Ks_star)
        if gap < tol:
            break
        K = (1.0 - damping) * K + damping * Ks
        K[0] = K0
    return dict(K=K, outer=outer, gap=gap)


def path_grades(out: dict, xis=(-0.2, -0.1, 0.1, 0.2), T=60) -> dict:
    """From tilted starts: the actor's K_t, the perfect-foresight K_t, and the fixed rule's K_t, in units of K*."""
    eco, key, actor = out["eco"], out["key"], out["actor"]
    rows = []
    for xi in xis:
        G0 = tilted_batch(eco, torch.tensor([xi], dtype=DT))[0].numpy()
        Ka, _ = actor_path(actor, eco, G0, T)
        Kp = perfect_foresight_path(key, G0, T=max(150, T))["K"][:T + 1]
        Kf = fixed_rule_path(eco, key, G0, T)
        rows.append(dict(xi=xi, K0=float(Ka[0]), K_actor=Ka.tolist(), K_pf=Kp.tolist(), K_fixed=Kf.tolist(),
                         gap_actor=float(np.abs(Ka - Kp).max() / eco.K_star),
                         gap_fixed=float(np.abs(Kf - Kp).max() / eco.K_star)))
    return dict(rows=rows, worst_actor=max(r["gap_actor"] for r in rows),
                worst_fixed=max(r["gap_fixed"] for r in rows))


def stationary_grade(out: dict, T=300) -> dict:
    """Push lambda* forward T periods under the actor: where its own law of motion settles."""
    eco = out["eco"]
    K, G = actor_path(out["actor"], eco, eco.lam.numpy(), T)
    return dict(K_T=float(K[-1]), K_star=eco.K_star, K_gap=float(K[-1] / eco.K_star - 1.0),
                tv=float(0.5 * np.abs(G - eco.lam.numpy()).sum()))


def summary(out: dict) -> dict:
    g = policy_grade(out["actor"], out["eco"])
    st = stationary_grade(out)
    pg = path_grades(out)
    return dict(seed=out["hist"]["cfg"]["seed"], seconds=round(out["hist"]["seconds"], 1),
                start=out["hist"]["grade"][0], **g, K_star=st["K_star"], K_T=st["K_T"], K_gap=st["K_gap"],
                tv=st["tv"], path_worst_actor=pg["worst_actor"], path_worst_fixed=pg["worst_fixed"],
                r_star=float(out["key"]["r"]))


def save(path, out):
    torch.save(dict(actor=out["actor"].state_dict(), critic=out["critic"].state_dict(),
                    target=out["target"].state_dict(), hist=out["hist"]), path)


def load(path, key: dict | None = None):
    rec = torch.load(path, weights_only=False)
    cfg = Config(**rec["hist"]["cfg"])
    key = answer_key(cfg) if key is None else key
    eco = Economy(key)
    actor, critic = Actor(eco, cfg.hidden), Critic(eco, cfg.hidden)
    target = Critic(eco, cfg.hidden)
    actor.load_state_dict(rec["actor"]); critic.load_state_dict(rec["critic"]); target.load_state_dict(rec["target"])
    return dict(actor=actor, critic=critic, target=target, eco=eco, key=key, hist=rec["hist"]), cfg


if __name__ == "__main__":
    import json
    import sys
    cfg = SMOKE if "smoke" in sys.argv else LIVE
    out = train(cfg)
    print(json.dumps(summary(out), indent=1, default=float))

In [ ]:
%%writefile l08_code/s08_deepham.py
"""Session 8 -- DeepHAM on Krusell and Smith (1998), in PyTorch, at CPU scale (8.B2; L08 Part C).

Ported on 2026-11-11 from the AI-ECON-2026 Session 4 lab (session04_deepham_ks.py), which was written from Han,
Yang and E, "DeepHAM: A Global Solution Method for Heterogeneous Agent Models with Aggregate Shocks" (QE 2026;
arXiv 2112.14377) and from the settings of the public repository's Krusell-Smith configuration. It is not a port
of that repository's code. Changes in the port: the economy is this course's s06_ks (6.B1; the same economy as
the source's), and the classical policy is rebuilt from 6.B1's coefficients in tools/figures/s06_lab_numbers.json
(ClassicalPolicy.from_ledger), not from a recorded file. Nothing else changed.

The economy is 6.B1's: Krusell and Smith (1998) in Den Haan, Judd and Juillard's (2010) parameterisation (log
utility, beta 0.99, alpha 0.36, delta 0.025, z in {0.99, 1.01}, unemployment 10% / 4%, a benefit of 15% of the
wage financed by a tax on the employed). A "distribution" here is a PANEL: E economies, each a sample of N
households (a_i, e_i) with continuous wealth and their own employment draws from the published chain
P(e' | e, z, z'); prices are marginal products at the panel's mean wealth K.

Three networks (8.B2, "DeepHAM in One Picture" and "Generalised Moments"):
  Q      the generalised moment, 1 -> 12 -> 12 -> 1 (tanh); Qbar = mean_i Q(a_i) is the one moment the others read;
  actor  (a_i, e_i, z, Qbar) -> a consumption share in (0, 1) of cash on hand, so saving a' >= 0 by construction;
  critic (a_i, e_i, z, Qbar) -> lifetime utility.
Three clocks (8.B2, "Three Clocks, and the Algorithm"):
  actor update   one Adam step per mini-batch of B pool rows, each unrolled T_unroll periods with fresh shocks;
                 the objective is household 0's discounted utility plus beta^T times the critic at the end; the
                 other N-1 households use the same network with the gradient stopped (fictitious play);
  pool rebuild   every U_pool = P / B updates (the pool is exhausted): every economy runs T_pool periods from its
                 stored state under the current actor, and every snap_every-th cross-section from snap_from on is
                 kept: P = E x n_snap rows, households shuffled within a row so that household 0 is a random member;
  critic refit   every F updates: every economy runs T_V periods from its stored state; the cross-sections at D
                 dates Delta_V apart are the rows, each household's target its realised discounted utility over the
                 next T_target periods; n_ep passes of mini-batches of B_V rows, one net.
  Where a run starts: the stored state of its economy, the burned-in cross-section until the first refit; from the
  first refit on, every run (pool or critic) replaces the stored state with its own last period.

Departures from the paper (each said in L08):
  1. scale: E, N, T_burn, T_pool, B, T_unroll, U, T_V, D, T_target, B_V, n_ep, F are a CPU-minute's worth; every
     field of Config carries the paper's value in a comment;
  2. one critic net, not three averaged;
  3. the actor is warm-started by fitting its consumption share to 6.B1's classical Krusell-Smith policy on the
     burned-in panel (the paper starts from a constant share 0.4); the burn-in itself runs under that policy;
  4. the generalised-moment net is trained with the actor only; the critic reads Qbar as a fixed input;
  5. prices use the model's unemployment rate u(z) for labour, not the panel's realised employment;
  6. the actor's learning rate is 1e-4 (the paper 4e-4), the critic's 3e-3 on standardised targets (the paper 1e-4).
Imports: numpy, torch, s06_ks.
"""
from __future__ import annotations

import copy
import json
import time
from dataclasses import dataclass, asdict, replace

import numpy as np
import torch
import torch.nn as nn

import s06_ks as ks

torch.set_num_threads(4)
DT = torch.float64

# ---------------------------------------------------------------------------------------------- the economy
P = ks.KSParams()
PZ_NP, PJ_NP = ks.ks_transition_matrix(P)             # 2x2 chain for z; 4x4 joint chain, state 2z + e
EMP_NP = np.zeros((2, 2, 2, 2))                       # P(e' | e, z, z') = joint / Pz
for _z in range(2):
    for _zn in range(2):
        for _e in range(2):
            for _en in range(2):
                EMP_NP[_z, _zn, _e, _en] = PJ_NP[2 * _z + _e, 2 * _zn + _en] / PZ_NP[_z, _zn]
PZ = torch.as_tensor(PZ_NP, dtype=DT)
EMP_STAY_U = torch.as_tensor(EMP_NP[:, :, :, 0], dtype=DT)     # P(e' = unemployed | e, z, z'), (2, 2, 2)
Z_LEVEL = torch.as_tensor(P.z, dtype=DT)
U_RATE = torch.as_tensor(P.u, dtype=DT)
A_SCALE = 50.0                                        # wealth is read by the networks as a / A_SCALE


def prices_t(K, zi):
    """Gross return, wage and tax at panel mean K (E,) in aggregate state zi (E,) long. Block B's formulas."""
    L = 1.0 - U_RATE[zi]
    kl = K / (P.l_bar * L)
    zl = Z_LEVEL[zi]
    r = P.alpha * zl * kl ** (P.alpha - 1.0)
    w = (1.0 - P.alpha) * zl * kl ** P.alpha
    tau = P.mu * U_RATE[zi] / (P.l_bar * L)
    return 1.0 - P.delta + r, w, tau


def cash_on_hand(a, e, zi):
    """a, e: (E, N); zi: (E,). Returns cash (E, N), K (E,), R (E,)."""
    K = a.mean(1)
    R, w, tau = prices_t(K, zi)
    inc = w[:, None] * torch.where(e == 1, ((1.0 - tau) * P.l_bar)[:, None].expand_as(a),
                                   torch.full_like(a, P.mu))
    return R[:, None] * a + inc, K, R


def draw_next(zi, e, gen):
    """One period of the exogenous chain: z' from Pz, then e' from P(e' | e, z, z')."""
    zn = torch.multinomial(PZ[zi], 1, generator=gen).squeeze(-1)
    u = torch.rand(e.shape, generator=gen, dtype=DT)
    p_unemp = EMP_STAY_U[zi[:, None], zn[:, None], e]            # (E, N)
    en = (u >= p_unemp).long()
    return zn, en


# ---------------------------------------------------------------------------------------------- the networks
class MLP(nn.Module):
    def __init__(self, n_in, width, n_out=1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, width), nn.Tanh(), nn.Linear(width, width), nn.Tanh(),
                                 nn.Linear(width, n_out))

    def forward(self, x):
        return self.net(x)


class DeepHAM(nn.Module):
    """The three networks and the panel arithmetic. Everything is float64 and runs on the CPU."""

    def __init__(self, hidden=24, q_hidden=12):
        super().__init__()
        self.Q = MLP(1, q_hidden)
        self.actor = MLP(4, hidden)
        self.critic = MLP(4, hidden)
        self.register_buffer("v_mu", torch.zeros((), dtype=DT))     # the critic is fitted on standardised targets
        self.register_buffer("v_sd", torch.ones((), dtype=DT))
        self.to(DT)

    def qbar(self, a):
        return self.Q((a / A_SCALE).unsqueeze(-1)).squeeze(-1).mean(1, keepdim=True)        # (E, 1)

    @staticmethod
    def features(a, e, zi, qbar):
        z = (2.0 * zi.to(DT) - 1.0)[:, None].expand_as(a)
        return torch.stack([a / A_SCALE, e.to(DT), z, qbar.expand_as(a)], dim=-1)

    def share(self, a, e, zi, qbar=None):
        if qbar is None:
            qbar = self.qbar(a)
        return torch.sigmoid(self.actor(self.features(a, e, zi, qbar)).squeeze(-1))

    def value(self, a, e, zi, qbar):
        return self.critic(self.features(a, e, zi, qbar)).squeeze(-1) * self.v_sd + self.v_mu

    def step(self, a, e, zi, gen, grad_hh0=False, share_fn=None):
        """One period of a panel: consumption today, the move to tomorrow. Returns (a', e', z', c, K)."""
        cash, K, _ = cash_on_hand(a, e, zi)
        if share_fn is None:
            sh = self.share(a, e, zi)
            if grad_hh0:                                  # fictitious play: only household 0 keeps its gradient
                sh = torch.cat([sh[:, :1], sh[:, 1:].detach()], dim=1)
        else:
            sh = share_fn(a, e, zi, cash, K)
        c = sh * cash
        a_next = cash - c
        zn, en = draw_next(zi, e, gen)
        return a_next, en, zn, c, K


# ---------------------------------------------------------------------------------------------- settings
@dataclass(frozen=True)
class Config:
    E: int = 32               # economies side by side (the paper: 384)
    N: int = 50               # households per economy (the paper: 50)
    T_burn: int = 300         # burn-in under the classical rule, from everyone at K_ss (the paper: 6,000)
    T_pool: int = 120         # periods of a pool run (the paper: 450)
    snap_from: int = 60       # first cross-section kept in a pool run (the paper: 250)
    snap_every: int = 4       # every 4th kept (the paper: 4)
    B: int = 64               # actor mini-batch, rows of the pool (the paper: 384)
    T_unroll: int = 150       # periods unrolled inside the actor's loss (the paper: 150; beta^149 = 0.22 to the critic)
    U: int = 250              # actor updates in all (the paper: 10,000)
    lr_actor: float = 1e-4    # (the paper: 4e-4; the lab starts from the classical policy, so it steps gently)
    T_V: int = 450            # periods of a critic run (the paper: 2,000)
    D: int = 4                # start dates of the critic's rows (the paper: 12)
    Delta_V: int = 100        # spacing of the dates (the paper: 100)
    T_target: int = 150       # horizon of a target, beta^150 = 0.22 (the paper: 800, beta^800 < 3.3e-4)
    B_V: int = 32             # critic mini-batch, rows (the paper: 128)
    n_ep: int = 120           # passes over the rows per fit (the paper: 201)
    F: int = 100              # actor updates between refits (the paper: 2,000)
    lr_critic: float = 3e-3   # on standardised targets, cosine-annealed over a fit (the paper: 1e-4)
    valid_share: float = 0.125  # rows set aside to check the fit (the paper: 512 of 4,608)
    hidden: int = 32          # actor and critic width (the lab's)
    q_hidden: int = 12        # moment-net width (the paper: 12)
    warm_steps: int = 3000    # warm start: fit the share to the classical policy (the lab's)
    log_every: int = 50
    seed: int = 0

    @property
    def n_snap(self) -> int:
        return len(range(self.snap_from, self.T_pool, self.snap_every))

    @property
    def P(self) -> int:
        return self.E * self.n_snap

    @property
    def U_pool(self) -> int:
        return max(1, self.P // self.B)


LIVE = Config()
RECORD = replace(LIVE, E=64, U=600, T_V=700, D=5, T_target=300, n_ep=150, F=200)
SMOKE = replace(LIVE, E=8, T_burn=40, T_pool=40, snap_from=20, B=8, T_unroll=10, U=12, T_V=60, D=2, Delta_V=20,
                T_target=30, n_ep=3, F=6, warm_steps=20, log_every=4)


# ---------------------------------------------------------------------------------------------- the classical policy
class ClassicalPolicy:
    """6.B1's Krusell-Smith household policy c(a, s, K) under its fitted rule kappa, read on a panel."""

    def __init__(self, kappa_bad, kappa_good, K_ss, settings: ks.KSSettings = ks.FULL):
        self.kappa = np.array([kappa_bad, kappa_good])
        s = settings
        self.a = ks.ks_grid(s.n_a, P.a_min, s.a_max, s.curv)
        self.logK = np.linspace(np.log(s.K_lo), np.log(s.K_hi), s.n_K)
        self.c, self.sweeps, _ = ks.solve_household_ks(self.kappa, P, self.a, self.logK, PJ_NP)
        self.K_ss = float(K_ss)

    @classmethod
    def from_ledger(cls, ks_section: dict):
        """ks_section: the "ks" section of tools/figures/s06_lab_numbers.json (6.B1's solution)."""
        sol = ks_section["solution"]
        return cls(sol["kappa_bad"], sol["kappa_good"], ks_section["diagnostics"]["K"]["mean"])

    def consumption(self, a, e, zi, K):
        """a, e: (E, N) numpy; zi, K: (E,). Linear in log K between nodes, linear in a; K clipped to the grid."""
        logK = np.clip(np.log(K), self.logK[0], self.logK[-1])
        pos = (logK - self.logK[0]) / (self.logK[1] - self.logK[0])
        m0 = np.clip(np.floor(pos).astype(int), 0, self.logK.size - 2)
        th = pos - m0
        out = np.empty_like(a)
        for i in range(a.shape[0]):
            for ee in range(2):
                sel = e[i] == ee
                if not sel.any():
                    continue
                s = 2 * int(zi[i]) + ee
                lo = np.interp(a[i, sel], self.a, self.c[:, s, m0[i]])
                hi = np.interp(a[i, sel], self.a, self.c[:, s, m0[i] + 1])
                out[i, sel] = (1.0 - th[i]) * lo + th[i] * hi
        return out

    def share_fn(self, a, e, zi, cash, K):
        c = self.consumption(a.numpy(), e.numpy(), zi.numpy(), K.numpy())
        c = np.minimum(c, 0.999 * cash.numpy())
        return torch.as_tensor(c / cash.numpy(), dtype=DT)


# ---------------------------------------------------------------------------------------------- simulation helpers
@torch.no_grad()
def simulate(model: DeepHAM, a, e, zi, T, gen, share_fn=None, keep=(), record_c=False):
    """Run a panel T periods. keep: periods whose cross-sections to return. Returns dict with the final state,
    the kept cross-sections, K path (E, T) and, if asked, every household's log consumption (E, T, N)."""
    E, N = a.shape
    K_path = torch.empty(E, T, dtype=DT)
    logc = torch.empty(E, T, N, dtype=DT) if record_c else None
    kept = {}
    for t in range(T):
        if t in keep:
            kept[t] = (a.clone(), e.clone(), zi.clone())
        a_next, en, zn, c, K = model.step(a, e, zi, gen, share_fn=share_fn)
        K_path[:, t] = K
        if record_c:
            logc[:, t] = torch.log(c)
        a, e, zi = a_next, en, zn
    return dict(a=a, e=e, z=zi, K=K_path, logc=logc, kept=kept)


def initial_panel(cfg: Config, K0, gen):
    """Everyone at K0; the aggregate state drawn; employment drawn at the state's unemployment rate."""
    zi = torch.multinomial(torch.full((cfg.E, 2), 0.5, dtype=DT), 1, generator=gen).squeeze(-1)
    u = torch.rand(cfg.E, cfg.N, generator=gen, dtype=DT)
    e = (u >= U_RATE[zi][:, None]).long()
    a = torch.full((cfg.E, cfg.N), float(K0), dtype=DT)
    return a, e, zi


def shuffle_households(a, e, gen):
    """Households permuted within each row, so that household 0 is a random member (the paper's reshuffle)."""
    perm = torch.argsort(torch.rand(a.shape, generator=gen, dtype=DT), dim=1)
    return torch.gather(a, 1, perm), torch.gather(e, 1, perm)


# ---------------------------------------------------------------------------------------------- training
def warm_start(model: DeepHAM, pol: ClassicalPolicy, a, e, zi, cfg: Config, gen, lr=5e-3):
    """The lab's warm start: the moment net is first fitted to Q(a) = a / A_SCALE, so that Qbar starts as the
    panel's mean wealth (Krusell and Smith's state); then the actor's share (and Q with it) is fitted to the
    classical policy's share on the burned-in panel, by relative consumption error."""
    with torch.no_grad():
        cash, K, _ = cash_on_hand(a, e, zi)
        target = pol.share_fn(a, e, zi, cash, K)
    opt_q = torch.optim.Adam(model.Q.parameters(), lr=3e-3)
    x = (a / A_SCALE).unsqueeze(-1)
    for _ in range(300):
        opt_q.zero_grad()
        lq = torch.mean((model.Q(x).squeeze(-1) - a / A_SCALE) ** 2)
        lq.backward(); opt_q.step()
    opt = torch.optim.Adam(list(model.actor.parameters()) + list(model.Q.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.warm_steps)
    mse = float("nan")
    for _ in range(cfg.warm_steps):
        opt.zero_grad()
        loss = torch.mean((model.share(a, e, zi) / target - 1.0) ** 2)      # relative consumption error
        loss.backward(); opt.step(); sched.step()
        mse = loss.item()
    return mse


def build_pool(model, stored, cfg: Config, gen, move_stored: bool):
    """Every economy runs T_pool periods from its stored state; every snap_every-th cross-section from snap_from
    on is a row. If move_stored, the run's last period replaces the stored state."""
    a, e, zi = stored
    keep = tuple(range(cfg.snap_from, cfg.T_pool, cfg.snap_every))
    out = simulate(model, a, e, zi, cfg.T_pool, gen, keep=keep)
    rows_a = torch.cat([out["kept"][t][0] for t in keep], 0)
    rows_e = torch.cat([out["kept"][t][1] for t in keep], 0)
    rows_z = torch.cat([out["kept"][t][2] for t in keep], 0)
    rows_a, rows_e = shuffle_households(rows_a, rows_e, gen)
    perm = torch.randperm(rows_a.shape[0], generator=gen)
    pool = (rows_a[perm], rows_e[perm], rows_z[perm])
    if move_stored:
        stored = (out["a"], out["e"], out["z"])
    return pool, stored


def actor_objective(model: DeepHAM, a, e, zi, cfg: Config, gen):
    """Household 0's discounted utility over T_unroll fresh periods plus beta^T times the critic at the end."""
    J = torch.zeros(a.shape[0], dtype=DT)
    disc = 1.0
    for _ in range(cfg.T_unroll):
        a, e, zi, c, _ = model.step(a, e, zi, gen, grad_hh0=True)
        J = J + disc * torch.log(c[:, 0])
        disc *= P.beta
    qbar = model.qbar(a)
    J = J + disc * model.value(a[:, :1], e[:, :1], zi, qbar)[:, 0]
    return J.mean()


def critic_rows(model, stored, cfg: Config, gen, move_stored: bool):
    """Runs of T_V periods from the stored states; rows at D dates; targets: realised discounted utility over
    T_target periods, every household its own."""
    a, e, zi = stored
    dates = [d * cfg.Delta_V for d in range(cfg.D)]
    assert dates[-1] + cfg.T_target <= cfg.T_V, "a date needs T_target future periods"
    out = simulate(model, a, e, zi, cfg.T_V, gen, keep=tuple(dates), record_c=True)
    disc = torch.as_tensor(P.beta ** np.arange(cfg.T_target), dtype=DT)
    rows = []
    for s in dates:
        ra, re, rz = out["kept"][s]
        target = (out["logc"][:, s:s + cfg.T_target] * disc[None, :, None]).sum(1)      # (E, N)
        rows.append((ra, re, rz, target))
    rows_a = torch.cat([r[0] for r in rows]); rows_e = torch.cat([r[1] for r in rows])
    rows_z = torch.cat([r[2] for r in rows]); rows_v = torch.cat([r[3] for r in rows])
    if move_stored:
        stored = (out["a"], out["e"], out["z"])
    return (rows_a, rows_e, rows_z, rows_v), stored


def fit_critic(model: DeepHAM, rows, cfg: Config, gen):
    """n_ep passes of mini-batches of B_V rows; a share of the rows set aside; returns (mse, held-out R^2)."""
    ra, re, rz, rv = rows
    R = ra.shape[0]
    perm = torch.randperm(R, generator=gen)
    n_valid = max(1, int(round(cfg.valid_share * R)))
    valid, train = perm[:n_valid], perm[n_valid:]
    with torch.no_grad():
        qbar = model.qbar(ra)                                          # Qbar of each row, a fixed input
        model.v_mu.fill_(rv[train].mean()); model.v_sd.fill_(rv[train].std() + 1e-12)
    opt = torch.optim.Adam(model.critic.parameters(), lr=cfg.lr_critic)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.n_ep)
    mse = float("nan")
    for _ in range(cfg.n_ep):
        order = train[torch.randperm(train.numel(), generator=gen)]
        for k in range(0, order.numel(), cfg.B_V):
            idx = order[k:k + cfg.B_V]
            opt.zero_grad()
            pred = model.value(ra[idx], re[idx], rz[idx], qbar[idx])
            loss = torch.mean(((pred - rv[idx]) / model.v_sd) ** 2)
            loss.backward(); opt.step()
            mse = loss.item()
        sched.step()
    with torch.no_grad():
        pred = model.value(ra[valid], re[valid], rz[valid], qbar[valid])
        resid = torch.mean((pred - rv[valid]) ** 2)
        r2 = 1.0 - resid / torch.var(rv[valid])
    return mse, float(r2)


def train(cfg: Config = LIVE, verbose: bool = True, pol: ClassicalPolicy | None = None):
    """Burn in, warm-start, fit the critic, then U actor updates with the three clocks. Returns (model, history)."""
    t0 = time.time()
    torch.manual_seed(cfg.seed)
    gen = torch.Generator().manual_seed(cfg.seed + 1)
    if pol is None:
        raise ValueError("pass pol = ClassicalPolicy.from_ledger(...), 6.B1's solution")
    model = DeepHAM(cfg.hidden, cfg.q_hidden)
    # 1. burn in under the classical rule, from everyone at K_ss, employment drawn at u(z)
    a, e, zi = initial_panel(cfg, pol.K_ss, gen)
    burn = simulate(model, a, e, zi, cfg.T_burn, gen, share_fn=pol.share_fn)
    stored = (burn["a"], burn["e"], burn["z"])
    # 2. warm start of the actor on the burned-in panel
    warm_mse = warm_start(model, pol, *stored, cfg, gen)
    hist = dict(objective=[], critic_mse=[], critic_r2=[], refit_at=[], pool_at=[], K_pool=[], warm_mse=warm_mse,
                K_burn=burn["K"][:, -1].mean().item())
    # 3. the first critic fit, before any actor update (the stored states do not move yet)
    rows, stored = critic_rows(model, stored, cfg, gen, move_stored=False)
    mse, r2 = fit_critic(model, rows, cfg, gen)
    hist["critic_mse"].append(mse); hist["critic_r2"].append(r2); hist["refit_at"].append(0)
    opt = torch.optim.Adam(list(model.actor.parameters()) + list(model.Q.parameters()), lr=cfg.lr_actor)
    update_init = False
    pool, stored = build_pool(model, stored, cfg, gen, move_stored=update_init)
    hist["pool_at"].append(0); hist["K_pool"].append(pool[0].mean().item())
    cursor = 0
    for u in range(cfg.U):
        if u > 0 and u % cfg.F == 0:                                   # critic refit; stored states move from now on
            update_init = True
            rows, stored = critic_rows(model, stored, cfg, gen, move_stored=True)
            mse, r2 = fit_critic(model, rows, cfg, gen)
            hist["critic_mse"].append(mse); hist["critic_r2"].append(r2); hist["refit_at"].append(u)
        if cursor + cfg.B > pool[0].shape[0]:                          # exhausted: rebuild the pool
            pool, stored = build_pool(model, stored, cfg, gen, move_stored=update_init)
            hist["pool_at"].append(u); hist["K_pool"].append(pool[0].mean().item())
            cursor = 0
        ra, re, rz = (x[cursor:cursor + cfg.B] for x in pool)
        cursor += cfg.B
        opt.zero_grad()
        J = actor_objective(model, ra, re, rz, cfg, gen)
        (-J).backward(); opt.step()
        hist["objective"].append(J.item())
        if verbose and (u % cfg.log_every == 0 or u == cfg.U - 1):
            print(f"update {u:5d} | objective {J.item():8.4f} | pools {len(hist['pool_at']):3d} "
                  f"| critic fits {len(hist['refit_at'])} (held-out R2 {hist['critic_r2'][-1]:.3f}) | {time.time() - t0:5.0f} s")
    hist["seconds"] = time.time() - t0
    hist["cfg"] = asdict(cfg)
    hist["stored"] = tuple(x.clone() for x in stored)
    return model, hist


# ---------------------------------------------------------------------------------------------- diagnostics
@torch.no_grad()
def fresh_panels(model, cfg: Config, pol: ClassicalPolicy, n_econ=16, T=300, seed=7, share_fn=None):
    """n_econ fresh economies from everyone at K_ss, burned in 100 periods under the policy to be judged."""
    gen = torch.Generator().manual_seed(seed)
    c2 = replace(cfg, E=n_econ)
    a, e, zi = initial_panel(c2, pol.K_ss, gen)
    out = simulate(model, a, e, zi, T, gen, share_fn=share_fn, record_c=True)
    return out, gen


@torch.no_grad()
def euler_errors(model: DeepHAM, cfg: Config, pol: ClassicalPolicy, n_econ=16, T=300, n_mc=16, seed=7,
                 classical=False):
    """Relative Euler error of the actor on a fresh panel: beta E[R' c / c'] - 1 by Monte Carlo over (z', e'),
    on households with slack saving. Log utility, so u'(c) = 1 / c. classical=True grades 6.B1's policy instead,
    on its own panel, with the same seeds: the benchmark the actor's number is read against."""
    if classical:
        out, gen = fresh_panels(model, cfg, pol, n_econ, T, seed, share_fn=pol.share_fn)
        share = lambda a_, e_, z_: pol.share_fn(a_, e_, z_, *cash_on_hand(a_, e_, z_)[:2])
    else:
        out, gen = fresh_panels(model, cfg, pol, n_econ, T, seed)
        share = model.share
    a, e, zi = out["a"], out["e"], out["z"]
    cash, K, _ = cash_on_hand(a, e, zi)
    sh = share(a, e, zi)
    c = sh * cash
    a_next = cash - c
    rhs = torch.zeros_like(c)
    for _ in range(n_mc):
        zn, en = draw_next(zi, e, gen)
        cash_n, _, Rn = cash_on_hand(a_next, en, zn)
        c_n = share(a_next, en, zn) * cash_n
        rhs += Rn[:, None] / c_n
    rhs /= n_mc
    err = P.beta * rhs * c - 1.0
    slack = a_next / cash > 1e-3
    ae = err[slack].abs()
    return dict(mean_abs=float(ae.mean()), p90_abs=float(torch.quantile(ae, 0.9)), share_slack=float(slack.to(DT).mean()),
                K_mean=float(out["K"][:, 100:].mean()), K_min=float(out["K"][:, 100:].min()),
                K_max=float(out["K"][:, 100:].max()), K_path=out["K"].numpy())


@torch.no_grad()
def welfare_vs_classical(model: DeepHAM, cfg: Config, pol: ClassicalPolicy, n_econ=16, T=300, seed=11):
    """Realised discounted utility of every household under the actor and under the classical policy, from the
    same initial panel with the same shocks; the gap as a consumption equivalent (log utility)."""
    disc = torch.as_tensor(P.beta ** np.arange(T), dtype=DT)
    outs = {}
    for name, fn in (("actor", None), ("classical", pol.share_fn)):
        out, _ = fresh_panels(model, cfg, pol, n_econ, T, seed, share_fn=fn)
        outs[name] = (out["logc"] * disc[None, :, None]).sum(1)         # (E, N)
    gap = outs["actor"].mean() - outs["classical"].mean()
    cev = float(torch.exp((1.0 - P.beta) * gap) - 1.0)
    return dict(utility_actor=float(outs["actor"].mean()), utility_classical=float(outs["classical"].mean()),
                cev_vs_classical=cev)


@torch.no_grad()
def share_plot_data(model: DeepHAM, cfg: Config, pol: ClassicalPolicy, seed=3):
    """Consumption share against wealth, at the panel of one fresh economy at K near K_ss, actor vs classical."""
    out, _ = fresh_panels(model, cfg, pol, n_econ=1, T=200, seed=seed)
    a, e, zi = out["a"], out["e"], out["z"]
    grid = torch.linspace(0.0, 120.0, 121, dtype=DT)
    rows = {}
    for ee in (0, 1):
        a_q = torch.cat([grid[None, :], a[:, :cfg.N - grid.numel()]], 1) if cfg.N > grid.numel() else grid[None, :]
        e_q = torch.full_like(a_q, ee, dtype=torch.long)
        cash, K, _ = cash_on_hand(a_q, e_q, zi)
        sh_a = model.share(a_q, e_q, zi)[0, :grid.numel()]
        sh_c = pol.share_fn(a_q, e_q, zi, cash, K)[0, :grid.numel()]
        rows[ee] = (sh_a.numpy(), sh_c.numpy())
    return dict(a=grid.numpy(), share=rows, z=int(zi[0]), K=float(a.mean()))


def objective_change(hist) -> dict:
    J = np.asarray(hist["objective"])
    n = max(1, len(J) // 10)
    return dict(first_tenth=float(J[:n].mean()), last_tenth=float(J[-n:].mean()),
                change=float(J[-n:].mean() - J[:n].mean()))


def summary(model, hist, cfg: Config, pol: ClassicalPolicy) -> dict:
    ee = euler_errors(model, cfg, pol)
    wf = welfare_vs_classical(model, cfg, pol)
    return dict(seed=cfg.seed, seconds=round(hist["seconds"], 1), warm_mse=hist["warm_mse"],
                euler_mean_abs=ee["mean_abs"], euler_p90_abs=ee["p90_abs"], share_slack=ee["share_slack"],
                K_mean=ee["K_mean"], K_min=ee["K_min"], K_max=ee["K_max"], K_ss_classical=pol.K_ss,
                critic_r2_last=hist["critic_r2"][-1], critic_r2_all=hist["critic_r2"],
                objective=objective_change(hist), **wf)


def save(path, model, hist):
    h = {k: v for k, v in hist.items() if k != "stored"}
    torch.save(dict(state_dict=model.state_dict(), history=h, stored=hist["stored"]), path)


def load(path):
    rec = torch.load(path, weights_only=False)
    cfg = Config(**rec["history"]["cfg"])
    model = DeepHAM(cfg.hidden, cfg.q_hidden)
    model.load_state_dict(rec["state_dict"])
    hist = dict(rec["history"]); hist["stored"] = rec["stored"]
    return model, hist, cfg


if __name__ == "__main__":
    import sys
    from pathlib import Path
    led = json.loads((Path(__file__).resolve().parents[3] / "tools/figures/s06_lab_numbers.json").read_text())
    cfg = SMOKE if "smoke" in sys.argv else LIVE
    pol = ClassicalPolicy.from_ledger(led["ks"])
    print(f"classical policy: {pol.sweeps} EGM sweeps; K_ss {pol.K_ss:.3f}")
    model, hist = train(cfg, verbose=True, pol=pol)
    print(json.dumps(summary(model, hist, cfg, pol), indent=1, default=float))

In [ ]:
import s08_ha_ac as H          # Part B
import s08_deepham as D        # Part C
DT = H.DT
print("imported", H.__name__, D.__name__)

---
## Part A — the answer key (6.A2, 6.B2)

The actor–critic will be graded against two things Session 6 can compute exactly.

1. **The stationary equilibrium** $(\lambda^*, K^*, r^*)$ and the household's rule there, by EGM and bisection on $r$
   (6.A2). We solve it on the grid the actor will use: $N = 40$ asset nodes up to $\bar a_N = 60$, packed near the
   borrowing limit. Session 6 used 200 nodes up to 150; the coarse grid moves $K^*$ by under one per cent.
2. **The perfect-foresight path.** Start the economy at a distribution $\Gamma_0 \neq \lambda^*$ with no shocks after.
   Prices then move deterministically back to the steady state, and 6.B2's loop (guess $K_t$, households backward,
   distribution forward, update) finds that path. A Markov rule $g(\varepsilon, a; \Gamma)$ that is right must
   reproduce it, because with no aggregate shocks the equilibrium path *is* what the rule implies from $\Gamma_0$.

The starting distributions are **tilts** of $\lambda^*$: $\Gamma \propto \lambda^* e^{\xi a / K^*}$, each productivity
column rescaled to keep its mass $\pi_q$. $\xi > 0$ moves wealth up; $\xi < 0$ moves it down.

In [ ]:
t0 = time.time()
key = H.answer_key(H.LIVE)                    # 6.A2's equilibrium on the 40-node histogram grid
eco = H.Economy(key)                          # the tensors every step needs
print(f"K* = {eco.K_star:.4f}   r* = {100 * eco.r_star:.3f} %   w* = {eco.w_star:.4f}   ({time.time() - t0:.1f} s)")
print(f"Session 6, 200 nodes up to 150:  K* = {LEDGER['s06_K_200']:.4f}   r* = {LEDGER['s06_r_pct_200']:.3f} %")
print(f"mass on the top five nodes: {float(eco.lam[-5:].sum()):.1e}  (the grid is wide enough)")

for xi in (-0.2, -0.1, 0.1, 0.2):
    G0 = H.tilted_batch(eco, torch.tensor([xi], dtype=DT))[0]
    K0 = float((G0.sum(1) * eco.grid).sum())
    print(f"tilt xi = {xi:+.1f}:  K0 / K* = {K0 / eco.K_star:.3f}   (ledger {LEDGER['tilt_K_over_Kstar'][f'{xi:+.1f}']})")

gate("A_K_star", rel(eco.K_star, LEDGER["K_star"]) < GATE["A_K_star_rel_max"],
     f"K* {eco.K_star:.10f} vs ledger {LEDGER['K_star']:.10f}")

In [ ]:
# One perfect-foresight path, from the richest tilt: 6.B2's loop, with no tax.
G0 = H.tilted_batch(eco, torch.tensor([0.2], dtype=DT))[0].numpy()
t0 = time.time()
pf = H.perfect_foresight_path(key, G0)
print(f"converged in {pf['outer']} passes, gap {pf['gap']:.1e}, {time.time() - t0:.1f} s")
plt.figure(figsize=(5.5, 3))
plt.plot(pf["K"][:80] / eco.K_star, label="perfect foresight from $\\xi=0.2$")
plt.axhline(1, color="grey", lw=0.8); plt.xlabel("year $t$"); plt.ylabel("$K_t/K^*$"); plt.legend(); plt.show()

---
## Part B — Aiyagari by actor–critic, the histogram as a state (8.B1)

**The problem the two networks solve.** A household's state is $(\varepsilon, a; \Gamma)$: its productivity, its
assets, and the whole distribution $\Gamma$, here a histogram of masses $\Gamma[i, q]$ on the 40 asset nodes and 7
productivity levels (280 numbers). Prices are $K(\Gamma)=\sum_{iq}\Gamma[i,q]\,\bar a_i$, $r = \alpha K^{\alpha-1}-\delta$,
$w=(1-\alpha)K^\alpha$. Tomorrow's distribution is $\Gamma_+ = \Gamma\,P(\Gamma;\theta)$: Young's lottery (6.A2) on everyone's
$a'$, then the productivity chain.

* **Actor** $g_\theta(\varepsilon,a;\Gamma)$: a saving share $s$ of cash on hand, $a' = s\,[(1+r)a+w\varepsilon]$, with
  $s = \min\{\max\{s^*(\varepsilon,a) + f_\theta,\,0\},\,0.99\}$. $s^*$ is the stationary EGM rule as a share
  (Part A), $f_\theta$ a network whose last layer starts at zero. **The first actor is the stationary rule**; the
  loop learns how $\Gamma$ should move it.
* **Critic** $V_\psi(\varepsilon,a;\Gamma)$, fitted to the one-step look-ahead with a **trailing copy** $\psi^-$
  (8.A1's "hold the target still").

**Departures from 8.B1**, all for size: Session 6's calibration (no labour choice); 40 nodes, not 140; 16 tilted
distributions per iteration; a 20-period rollout; the actor starts from the one stationary rule rather than being
warm-started from many.

### B1 — the state and the two networks

In [ ]:
actor0 = H.Actor(eco, H.LIVE.hidden)          # f = 0 at the start: this IS the stationary rule
critic0 = H.Critic(eco, H.LIVE.hidden)
n_a = sum(p.numel() for p in actor0.parameters()); n_c = sum(p.numel() for p in critic0.parameters())
print(f"actor parameters {n_a:,}   critic parameters {n_c:,}   inputs: 2 per node + {eco.N * eco.Q} for Gamma")

# The networks read Gamma through its relative deviation from lambda*: d = (Gamma - lambda*) / (lambda* + 1/NQ).
G = H.tilted_batch(eco, torch.tensor([-0.2, 0.0, 0.2], dtype=DT))     # three economies
ap = H.policy_on_nodes(actor0, eco, G).detach()                       # (3, N, Q): a' at every node
K, r, w = eco.prices(G)
for x, k_, r_ in zip((-0.2, 0.0, 0.2), K, r):
    print(f"xi {x:+.1f}: K = {float(k_):.3f}, r = {100 * float(r_):.2f} %")
print("max |a' - EGM| at lambda*:", float((ap[1] - eco.ap).abs().max()))

The starting actor already responds to $\Gamma$ in one way: through **today's** prices, since its saving share
multiplies today's cash on hand. What it cannot know is how prices will move **tomorrow** as the distribution
returns to $\lambda^*$. That is what the loop must teach it.

### B2 — $\Gamma_+$ through the lottery, with a gradient
The rollout penalty and the actor's loss differentiate *through* the law of motion. In `push_batch` the node index
$j$ is a fixed integer, but the weight sent up, $(a'-\bar a_j)/(\bar a_{j+1}-\bar a_j)$, is smooth in $a'$, so
$\Gamma_+$ is differentiable in the policy. Check it against a finite difference: shift every $a'$ by $s$ and
differentiate tomorrow's mean wealth and mean squared wealth.

In [ ]:
G = eco.lam[None]
def moments_plus(s):
    Gp = H.push_batch(G, (eco.ap + 0.01 + s)[None], eco)        # +0.01 keeps every a' off the borrowing limit
    m = Gp.sum(-1)[0]
    return (m * eco.grid).sum(), (m * eco.grid ** 2).sum()
s = torch.zeros((), dtype=DT, requires_grad=True)
K1, M2 = moments_plus(s)
dK = torch.autograd.grad(K1, s, retain_graph=True)[0]; dM = torch.autograd.grad(M2, s)[0]
h = 1e-6
fK = (moments_plus(h)[0] - moments_plus(-h)[0]) / (2 * h); fM = (moments_plus(h)[1] - moments_plus(-h)[1]) / (2 * h)
print(f"dK+/ds   autograd {float(dK):.10f}   finite difference {float(fK):.10f}")
print(f"dM2+/ds  autograd {float(dM):.10f}   finite difference {float(fM):.10f}")
print("dK+/ds = 1: the lottery keeps the mean, so moving everyone up by s moves K+ by s.")
gate("B2_lottery_gradient", max(abs(float(dK - fK)), abs(float(dM - fM)) / abs(float(fM))) < GATE["B2_autograd_vs_fd_max"])

### B3 — the critic step (8.B1, "The Critic")
The label is $b = u(c) + \beta\sum_{q'}\Pi_{qq'}V_{\psi^-}(\varepsilon_{q'}, a'; \Gamma_+)$, computed under `no_grad`
from the trailing copy, so it is data. The loss is the mean squared gap in the critic's own units; after the step,
$\psi^-\leftarrow(1-\tau)\psi^-+\tau\psi$ with $\tau = 0.02$. Read `bellman_pieces` and `critic_step` in `s08_ha_ac.py`.

### B4 — the actor step (8.B1, "The Actor")
$\mathcal L_g = -J + \kappa\,\overline{B} + \nu_K\cdot\frac1T\sum_t (K_t/K^*-1)^2$.

* $J$ is the mean of $u(c)+\beta\,C(a')$ over every node of every tilted $\Gamma$. Its slope in $a'$ uses the
  **envelope condition**, $\beta\sum_{q'}\Pi_{qq'}(1+r_+)u'(c_+)$, so the gradient of $J$ is the Euler residual
  times $\partial a'/\partial\theta$ (`C_eff` in `bellman_pieces`).
* $B$ is a hinge barrier that is zero wherever $c>0$.
* The **rollout penalty** pushes $\lambda^*$ forward $T$ periods under the actor's own law of motion, inside the graph,
  and penalises any drift of $K_t$ from $K^*$. Without it, tiny policy errors compound: a stationary distribution
  amplifies a saving error by roughly $1/(1-\beta(1+r))$.

Look at one step's pieces on a batch:

In [ ]:
cfg = H.LAB
target0 = H.copy.deepcopy(critic0)
Gb = H.tilted_batch(eco, torch.tensor([-0.2, 0.2], dtype=DT))
pc = H.bellman_pieces(actor0, target0, eco, Gb, cfg, grad_actor=True)
print("consumption > 0 everywhere:", bool((pc["c"] > 0).all()), "| barrier:", float(pc["B"].mean()))
print("r today:", [f"{100 * float(x):.2f} %" for x in pc["r"]])
_, rp, _ = eco.prices(pc["Gp"])
print("r tomorrow under this actor:", [f"{100 * float(x):.2f} %" for x in rp], "  <- the household must foresee this")
print("rollout penalty of the starting actor:", float(H.rollout_penalty(actor0, eco, cfg)))

### B5 — the loop (8.B1, "The Algorithm")
Pre-fit the critic to the starting actor, then per iteration: a fresh batch of tilted distributions, `n_critic = 8`
critic steps, `n_actor = 2` actor steps. The log prints, at $\Gamma=\lambda^*$, the actor's mean distance from the
EGM rule (in units of $K^*$) and its mean Euler residual where the rule is interior.

In [ ]:
if USE_RECORDED:
    out, cfg_run = H.load(ASSETS / "recorded_ha_ac.pt", key=key)
    print(f"loaded the reference run (LIVE, seed 0): {out['hist']['seconds']:.0f} s when it was trained")
else:
    out = H.train(H.LAB, key=key)                 # about a minute and a half
    cfg_run = H.LAB
hist = out["hist"]
it = [g["iteration"] for g in hist["grade"]]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].semilogy(hist["critic"]); ax[0].set_title("critic loss (pre-fit, then the loop)"); ax[0].set_xlabel("step")
ax[1].plot(it, [g["euler_mean"] for g in hist["grade"]], "o-")
ax[1].axhline(hist["grade"][0]["euler_mean"], color="grey", ls="--", label="the stationary rule")
ax[1].set_title("mean Euler residual at $\\lambda^*$"); ax[1].set_xlabel("iteration"); ax[1].legend(); plt.tight_layout(); plt.show()

### B6 — the grades
Three tests, from cheapest to hardest.

1. **At $\Gamma=\lambda^*$**: the actor against the stationary EGM rule, and its Euler residual.
2. **Where its own law of motion settles**: push $\lambda^*$ forward 300 periods under the actor; $K_T$ should stay at $K^*$.
3. **Paths from tilted starts**: $K_t$ under the actor against the perfect-foresight path, beside two simpler rules:
   the **fixed rule** (every household keeps the stationary $a'$ whatever the prices) and the **starting actor**
   (the stationary saving share of *today's* cash).

In [ ]:
g = H.policy_grade(out["actor"], eco)
st = H.stationary_grade(out)
print(f"at lambda*: mean |a' - EGM| = {g['ap_gap_mean']:.4f} K*,  Euler residual {g['euler_mean']:.2e}"
      f"  (the stationary rule's own: {hist['grade'][0]['euler_mean']:.2e})")
print(f"after 300 periods: K_T / K* - 1 = {st['K_gap']:+.4f},  total variation from lambda* {st['tv']:.3f}")

pg = H.path_grades(out)                                      # xi = -0.2, -0.1, 0.1, 0.2; 60 years
start = dict(out, actor=actor0)
pg0 = H.path_grades(start)
print(f"\nworst |K_t - K_t^pf| / K* over 60 years and four tilts:")
print(f"  fixed rule      {pg['worst_fixed']:.4f}")
print(f"  starting actor  {pg0['worst_actor']:.4f}")
print(f"  trained actor   {pg['worst_actor']:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2), sharey=False)
for a_, row, row0 in zip(ax, (pg["rows"][0], pg["rows"][-1]), (pg0["rows"][0], pg0["rows"][-1])):
    a_.plot(np.array(row["K_pf"]) / eco.K_star, "k-", lw=2, label="perfect foresight")
    a_.plot(np.array(row["K_fixed"]) / eco.K_star, ":", label="fixed rule")
    a_.plot(np.array(row0["K_actor"]) / eco.K_star, "--", label="starting actor")
    a_.plot(np.array(row["K_actor"]) / eco.K_star, "-", label="trained actor")
    a_.set_title(f"from $\\xi={row['xi']:+.1f}$"); a_.set_xlabel("year"); a_.axhline(1, color="grey", lw=0.6)
ax[0].set_ylabel("$K_t/K^*$"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"reference: starting actor {LEDGER['start_path_worst']:.4f}, fixed rule {LEDGER['live_seed0']['path_worst_fixed']:.4f}")
print(f"\nreference run (LIVE, seed 0): Euler {LEDGER['live_seed0']['euler_mean']:.2e}, path gap "
      f"{LEDGER['live_seed0']['path_worst_actor']:.4f} vs the fixed rule's {LEDGER['live_seed0']['path_worst_fixed']:.4f}")
print(f"median over {LEDGER['live_seeds']} seeds: Euler {LEDGER['live_median']['euler_mean']:.2e}, "
      f"path gap {LEDGER['live_median']['path_worst_actor']:.4f}")
gate("B5_euler_improves", g["euler_mean"] < hist["grade"][0]["euler_mean"],
     f"{g['euler_mean']:.2e} < {hist['grade'][0]['euler_mean']:.2e}")
gate("B5_K_pinned", abs(st["K_gap"]) < GATE["B5_K_gap_max"], f"|K_T/K* - 1| = {abs(st['K_gap']):.4f}")
gate("B6_paths", pg["worst_actor"] < GATE["B6_path_ratio_max"] * pg["worst_fixed"],
     f"{pg['worst_actor']:.4f} < {GATE['B6_path_ratio_max']} x {pg['worst_fixed']:.4f}")
if USE_RECORDED:
    gate("R_recorded_B", rel(g["euler_mean"], LEDGER["live_seed0"]["euler_mean"]) < GATE["R_recorded_rel_max"]
         and rel(pg["worst_actor"], LEDGER["live_seed0"]["path_worst_actor"]) < GATE["R_recorded_rel_max"])

**What to read off.** The fixed rule ignores prices and misses the path most. Saving a share of today's cash on
hand closes part of the gap, because the household responds to today's interest rate. The trained actor closes
most of the rest: reading $\Gamma$, it has learned that a rich economy's high capital means low interest rates *for
some years*, not just today, and saves accordingly. The Euler residual at $\lambda^*$ also falls below the
stationary rule's: on 40 nodes the EGM rule is itself an approximation, and the actor is trained on the Euler
equation directly.

**What it does not show.** Every grade here has an answer key because the economy has no aggregate shocks. 8.B1's
point is the case without one; there the grades are residuals and aggregates, and the answer key is gone.

---
## Part C — Krusell–Smith by DeepHAM (6.B1, 8.B2)

Krusell–Smith adds aggregate shocks, so there is no perfect-foresight path to grade against. 6.B1 solved it with a
forecast rule $\log K' = \kappa_0(z) + \kappa_1(z)\log K$; DeepHAM replaces both the rule and the household's policy
with networks, and reads the distribution through a **panel** of $N = 50$ households.

* **Generalised moment** $Q(a)$, a small network; the state the others read is $\bar Q = \frac1N\sum_i Q(a_i)$.
  It starts as $Q(a) = a/50$, so $\bar Q$ starts as mean wealth, Krusell and Smith's moment.
* **Actor** $(a_i, e_i, z, \bar Q)\mapsto$ a consumption share in $(0,1)$ of cash on hand.
* **Critic** $(a_i, e_i, z, \bar Q)\mapsto$ lifetime utility, fitted to realised discounted utility.
* **Fictitious play.** In the actor's objective only household 0 keeps its gradient; the other 49 use the same network
  with the gradient stopped. Each household optimises taking the others' rule as given.
* **Three clocks** (8.B2): an actor update per mini-batch of pool rows; a pool rebuild when the pool runs out; a
  critic refit every $F$ updates.

### C1 — 6.B1's policy, rebuilt from its forecast rule

In [ ]:
t0 = time.time()
pol = D.ClassicalPolicy(LEDGER["ks_kappa_bad"], LEDGER["ks_kappa_good"], LEDGER["ks_K_mean"])
print(f"6.B1's household under its forecast rule: {pol.sweeps} EGM sweeps, {time.time() - t0:.1f} s;  "
      f"mean K in 6.B1's simulation {pol.K_ss:.2f}")
cfg_dh = D.LIVE
print(f"DeepHAM's settings: E = {cfg_dh.E} economies x N = {cfg_dh.N} households; pool of {cfg_dh.P} rows; "
      f"{cfg_dh.U} actor updates, critic refit every {cfg_dh.F}; unrolled {cfg_dh.T_unroll} periods")

### C2–C4 — warm start, critic, actor, the three clocks
`train` burns the panel in under 6.B1's policy, fits the actor's share to it (the warm start), fits the critic,
then runs the actor updates. Read `actor_objective` (C2: household 0's discounted utility over `T_unroll` periods
plus $\beta^{T}$ times the critic), `critic_rows` and `fit_critic` (C3), and the loop in `train` (C4).

In [ ]:
if USE_RECORDED:
    model, hist_dh, cfg_dh = D.load(ASSETS / "recorded_deepham.pt")
    print(f"loaded the reference run (LIVE, seed 0): {hist_dh['seconds']:.0f} s when it was trained")
else:
    model, hist_dh = D.train(cfg_dh, verbose=True, pol=pol)      # a few minutes
J = np.asarray(hist_dh["objective"])
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(J, lw=0.6); ax[0].set_title("actor objective per update"); ax[0].set_xlabel("update")
for u in hist_dh["refit_at"]:
    ax[0].axvline(u, color="grey", lw=0.6, ls="--")
ax[1].bar(range(len(hist_dh["critic_r2"])), hist_dh["critic_r2"]); ax[1].set_title("critic's held-out $R^2$ per fit")
plt.tight_layout(); plt.show()

### C5 — the grades, against 6.B1
1. **Euler errors** on fresh panels: $\beta\,\mathbb E[R'c/c'] - 1$ by Monte Carlo over $(z', e')$, for households
   who save. 6.B1's policy is graded the same way, on the same seeds.
2. **Capital**: the mean of $K_t$ on fresh panels against 6.B1's simulation.
3. **Welfare**: realised discounted utility under the actor and under 6.B1's policy, from the same panels and shocks,
   as a consumption equivalent.
4. **The learned moment**: is $Q(a)$ still close to $a/50$?

In [ ]:
ee = D.euler_errors(model, cfg_dh, pol)
ee_c = D.euler_errors(model, cfg_dh, pol, classical=True)
wf = D.welfare_vs_classical(model, cfg_dh, pol)
print(f"mean |Euler error|: DeepHAM {ee['mean_abs']:.4f}   6.B1's policy {ee_c['mean_abs']:.4f}")
print(f"mean K on fresh panels: DeepHAM {ee['K_mean']:.2f}   6.B1's policy {ee_c['K_mean']:.2f}"
      f"   (6.B1's long simulation: {pol.K_ss:.2f})")
print(f"consumption equivalent of DeepHAM's policy against 6.B1's: {100 * wf['cev_vs_classical']:+.3f} %")
print(f"reference median over {LEDGER['dh_seeds']} seeds: |Euler| {LEDGER['dh_median']['euler_mean_abs']:.4f}, "
      f"K {LEDGER['dh_median']['K_mean']:.2f}, CEV {100 * LEDGER['dh_median']['cev_vs_classical']:+.3f} %")

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3))
ax[0].plot(ee["K_path"][:4].T, lw=0.8); ax[0].axhline(pol.K_ss, color="k", ls="--", lw=0.8)
ax[0].set_title("K_t on four fresh panels"); ax[0].set_xlabel("quarter")
aa = torch.linspace(0, 150, 151, dtype=DT)
with torch.no_grad():
    q = model.Q((aa / D.A_SCALE)[:, None]).squeeze(-1)
ax[1].plot(aa, q, label="learned $Q(a)$"); ax[1].plot(aa, aa / D.A_SCALE, "--", label="$a/50$ (the start)")
ax[1].set_xlabel("wealth $a$"); ax[1].legend(); plt.tight_layout(); plt.show()

gate("C_K", abs(ee["K_mean"] / pol.K_ss - 1) < GATE["C_K_rel_gap_max"], f"K {ee['K_mean']:.2f} vs {pol.K_ss:.2f}")
gate("C_euler", ee["mean_abs"] < GATE["C_euler_mean_max"], f"{ee['mean_abs']:.4f}")
gate("C_cev", abs(wf["cev_vs_classical"]) < GATE["C_abs_cev_max"], f"{wf['cev_vs_classical']:+.5f}")
if USE_RECORDED:
    gate("R_recorded_C", rel(ee["mean_abs"], LEDGER["dh_seed0"]["euler_mean_abs"]) < GATE["R_recorded_rel_max"]
         and rel(ee["K_mean"], LEDGER["dh_seed0"]["K_mean"]) < GATE["R_recorded_rel_max"])

**What to read off.** Starting from 6.B1's policy and keeping no forecast rule, DeepHAM stays close to 6.B1's
answer: mean capital within a few per cent of 6.B1's policy on the same panels, and welfare within a tenth of a per
cent of consumption. Its Euler errors are larger than 6.B1's (both are Monte Carlo estimates on panels of 50). A
network trained for a few minutes does not beat a carefully solved classical model here; the point is that the same
machinery runs where no forecast rule is available.

---
## Exercises

1. **Without the rollout penalty.** Train Part B with `H.replace(H.LAB, nu_K=0.0)`. Where does $K_T$ settle after
   300 periods? (The reference run without it: $|K_T/K^*-1| =$ `LEDGER["noroll_K_gap"]`.) Explain the drift with the
   amplification $1/(1-\beta(1+r))$.
2. **Only one distribution.** Train with `xi_max=0.0`, so the actor sees only $\lambda^*$. Re-run B6. Which grade
   gets worse, and why does the grade at $\lambda^*$ not notice?
3. **Out of range.** The batch tilts span $\xi\in[-0.2, 0.2]$. Grade the trained actor on paths from $\xi=\pm0.5$
   (`H.path_grades(out, xis=(-0.5, 0.5))`). How far does the network extrapolate?
4. **No trailing copy.** Set `tau=1.0`, so the critic's label is read from the critic itself. Compare the critic loss
   curve with B5's. Why is a label read from the network being fitted harder to hit?
5. **The critic's slope.** In `bellman_pieces`, replace `C_eff` by `C` in the actor's $J$ (one line; edit
   `l08_code/s08_ha_ac.py` and re-import). The actor's gradient then uses the critic's slope in $a'$ instead of the
   envelope condition. What happens to the Euler residual?
6. **Where the actor differs.** Plot $a'$ of the trained actor minus the starting actor at $\xi=+0.2$ and $\xi=-0.2$,
   by asset node, for the median productivity. Who saves more in a rich economy, and why?
7. **Away from $\lambda^*$.** `policy_grade` grades at $\lambda^*$ only. Write the same Euler residual at a tilted
   $\Gamma$ (copy the function and change its first line). Is the actor as accurate there?
8. **Fewer households.** Train DeepHAM with `D.replace(D.LIVE, N=10)`. What happens to $K$'s volatility across fresh
   panels and to the Euler errors? Why does a panel of 10 make $\bar Q$ noisy?
9. **A shorter unroll.** Set `T_unroll=30`. The critic now carries $\beta^{30}$ of the objective instead of $\beta^{150}$.
   Compare the welfare gap with C5's.
10. **The moment.** Fit $\bar Q$ on fresh panels by OLS on mean wealth. If $R^2\approx1$, DeepHAM has rediscovered
    Krusell and Smith's moment; discuss what it would take for a second moment to matter.

## Takeaways

1. With the distribution as a state, the household's rule can be graded against an exact perfect-foresight path,
   and it beats both a rule that ignores prices and one that sees only today's.
2. The law of motion is differentiable: Young's lottery carries a gradient from tomorrow's distribution to today's policy.
3. A rollout penalty inside the graph pins aggregate capital; without it, small policy errors compound.
4. DeepHAM reaches 6.B1's Krusell–Smith answer from a panel and a learned moment, with no forecast rule.